In [ ]:
%run scripts/verify_environment.py

verify_environment()

In [ ]:

from datetime import datetime, timedelta
from getpass import getpass
from urllib.parse import urlparse

admin_rdm_url = 'https://admin.staging.rdm.example.com'
rdm_url = 'https://staging.rdm.example.com'
myproject_url = 'https://staging.rdm.example.com/myprojects'
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
idp_name_institutional_admin = None
idp_username_institutional_admin = None
idp_password_institutional_admin = None
rdm_project_name = 'TEST-METADATA-未病データベース-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
default_result_path = None
close_on_fail = False
transition_timeout = 60000
ignore_https_errors = False

# WEKO情報
weko_url = None
weko_admin_email = None
weko_admin_password = None
weko_index_name = None
oauth_application_name = 'TEST-APP-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
oauth_client_id = None
oauth_client_secret = None

# WEKO SWORD API settings
weko_docker_compose_path = None
sword_mapping_id = 51000

_release = datetime.now() + timedelta(days=365)
scheduled_release_date = _release.strftime('%Y/%m/%d')
weko_scheduled_release_date = _release.strftime('%Y-%m-%d')
project_url = None
target_storage_name = "NII Storage"

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if idp_username_institutional_admin is None:
    idp_username_institutional_admin = input(prompt=f'Username for {idp_name_institutional_admin}')
if idp_password_institutional_admin is None:
    idp_password_institutional_admin = getpass(prompt=f'Password for {idp_username_institutional_admin}@{idp_name_institutional_admin}')

if weko_url is None:
    weko_url = input('WEKO URL: ')
if weko_admin_email is None:
    weko_admin_email = input('WEKO user email: ')
if weko_admin_password is None:
    weko_admin_password = getpass('WEKO user password: ')
if weko_index_name is None:
    weko_index_name = input('WEKO index: ')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# 未病データベース_プロジェクトメタデータ

- サブシステム名: アドオン
- ページ/アドオン: Metadata
- 機能分類: メタデータ入力
- シナリオ名: 未病データベース_プロジェクトメタデータ
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

import scripts.metadata_mibyou_db
importlib.reload(scripts.metadata_mibyou_db)
from scripts.metadata_mibyou_db import FileMibyouDbMetadataForm

await init_pw_context(close_on_fail=False, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
import time

async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    # GRDMのボタンが表示されることを確認
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)
    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から指定されたプロジェクトをクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()
    global project_url
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

「アドオンを構成」のパネル内に「Metadata」の行が表示されること

In [ ]:
addon_name = 'Metadata'
addon_page_url = None

async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()

    await expect(page.locator(f'//h4[@class="addon-title"][normalize-space(.)="{addon_name}"]')).to_be_visible(timeout=transition_timeout)
    global addon_page_url
    addon_page_url = page.url

await run_pw(_step)

## WEKOのアプリケーション設定のURLを開く

WEKO3のトップ画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url + '/account/settings/applications/')
    await expect(page.locator('//input[@name = "email"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## WEKOアカウントを入力し、「Log in」をクリックする

WEKO3のアプリケーション設定の画面が表示されること


In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(weko_admin_email)
    await page.locator('//input[@name = "password"]').fill(weko_admin_password)
    await page.locator('//button[@type = "submit"]').click()
    await expect(page.locator('//strong[contains(text(), "Developer Applications")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「New application」をクリックする

New OAuth Applicationフォームが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "New application")]').click()
    await expect(page.locator('//strong[contains(text(), "New OAuth Application")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## アプリケーション情報を入力し、「登録」をクリックする

フォームには以下の情報を入力する。Client IDとClient Secretが表示されること

- 名前: TEST-WEKO-APP-YYYYMMDDHHMM(実施時刻)
- 記述: TEST-WEKO-APP-YYYYMMDDHHMM(実施時刻)
- Website URL: (GakuNin RDMシステムのURL)
- Redirect URIs: (GakuNin RDMシステムのURL)/oauth/callback/weko/(WEKO3リポジトリのホスト名: 例えばweko3.rdm.example.com)/
- Client type: Confidential

In [ ]:
base_url = rdm_url[:-1] if rdm_url.endswith('/') else rdm_url
redirect_uri = f"{base_url}/oauth/callback/weko/{urlparse(weko_url).hostname}/"
application_description = oauth_application_name
print('Redirect URI:', redirect_uri)


In [ ]:
async def _step(page):
    await page.locator('//input[@name = "name"]').fill(oauth_application_name)
    await page.locator('//textarea[@name = "description"]').fill(application_description)
    # localhost:port は websiteに指定できないので置き換え
    website_url = rdm_url.replace('localhost', 'example.com')
    await page.locator('//input[@name = "website"]').fill(website_url)
    await page.locator('//textarea[@name = "redirect_uris"]').fill(redirect_uri)
    await page.locator('//button[@type = "submit"]').click()
    await expect(page.locator('//strong[text() = "Client ID"]')).to_be_visible(timeout=transition_timeout)
    global oauth_client_id, oauth_client_secret
    oauth_client_id = (await page.locator('//strong[text() = "Client ID"]/../code').inner_text()).strip()
    oauth_client_secret = (await page.locator('//strong[text() = "Client Secret"]/../code').inner_text()).strip()
    print('Retrieved client credentials from WEKO')

await run_pw(_step)


## プロジェクトダッシュボードでのテスト用のSWORD Client登録

プロジェクトダッシュボードでのテスト用に、WEKOにSWORD Clientを登録する。

In [ ]:
import subprocess
import textwrap

async def _step(page):
    if weko_docker_compose_path:
        code = textwrap.dedent(f'''
from weko_swordserver.api import SwordClient
from weko_swordserver.models import SwordClientModel as M
obj = SwordClient.register(
    client_id="{oauth_client_id}",
    registration_type_id=M.RegistrationType.DIRECT,
    mapping_id={sword_mapping_id},
    active=True,
    duplicate_check=False,
)
print(f"Registered SWORD client: {{obj.client_id}}")
''').strip()
        subprocess.run(
            ['docker', 'compose', '-f', weko_docker_compose_path, 'exec', '-T', 'web',
            'invenio', 'shell', '-c', code],
            check=True
        )
    else:
        await page.goto(weko_url + '/admin/swordapi/jsonld/')
        await expect(page.locator('//h4[text() = "JSON-LD"]')).to_be_visible(timeout=transition_timeout)

        await page.get_by_role('link', name='Create').click()
        await page.locator('#application').select_option(label=f'{oauth_application_name}')
        await page.locator('#mapping').select_option(value=f'{sword_mapping_id}')
        await expect(page.locator('#save_button')).to_be_enabled(timeout=transition_timeout)
        await page.locator('#save_button').click()
        await expect(page.locator("table.model-list")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザの別タブで、GakuNinRDMの管理者画面を開く。ステージング：https://admin.staging.example.com.vn/

GakuNinRDMの管理者画面のログイン画面が開くこと

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## GakuNinRDMの管理者画面のログイン画面で、プルダウンから「GakuNin　RDM Idp」を選択して「選択」ボタンをクリックする

Username、Password入力画面が開くこと
※本手順の確認内容は自動検証できないため、次手順の中で結果を一括確認します。

In [ ]:
async def _step(page):
    pass

await run_pw(_step)

## Username、Passwordに、機関管理者権限のアカウントアカウントとパスワードを入力して、Loginをクリックする

RDM Adminのメインページが開くこと

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_institutional_admin, idp_username_institutional_admin, idp_password_institutional_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 左側メニューよりRDM Addonsをクリックする

RDM Addonsのページが開くこと

In [ ]:
async def _step(page):
    await page.locator('a[href="/addons/"]').click()
    await page.wait_for_url("**/addons/**", timeout=transition_timeout)

    await expect(page.locator('h2:has-text("アドオン利用制御")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## RDM Addonsのページのリストから、JAIRO Cloud選択しチェックボックスをチェックし、追加ボタンをクリックする

Configure a JAIRO Cloud applicationの画面が開くこと

In [ ]:
async def _step(page):
    checkbox = page.locator('input[type="checkbox"][data-addon-short-name="weko"]')
    if not await checkbox.is_checked():
        await checkbox.check()
    await page.locator('button.btn.btn-success:has-text("追加")').click()

    await expect(page.locator('#wekoInputHost')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## Configure a JAIRO Cloud applicationの画面で以下の項目を入力して、保存をクリックすること 
・JAIRO Cloud Display Name 　→ ams-ir  
・JAIRO Cloud URL 　 → http://ams.ir.example
・JAIRO Cloud OAuth Client ID 　→ (個別に情報共有)  
・JAIRO Cloud OAuth Client Secret 　→ (個別に情報共有)  

RDM AddonsのページのリストのJAIRO CloudにOauth Application ams-ir が追加されること

In [ ]:
async def _step(page):
    time.sleep(2)
    modal = page.locator('#wekoInputHost')
    await modal.locator('input[name="weko_name"]').fill(oauth_application_name)
    await modal.locator('input[name="weko_url"]').fill(weko_url)
    await modal.locator('input[name="weko_oauth_client_id"]').fill(oauth_client_id)
    await modal.locator('input[name="weko_oauth_client_secret"]').fill(oauth_client_secret)
    await page.keyboard.press('Tab')

    await expect(modal.locator('button.btn.btn-success:has-text("保存")')).to_be_enabled(timeout=transition_timeout)
    await modal.locator('button.btn.btn-success:has-text("保存")').click()
    time.sleep(2)

    await expect(page.locator('#weko-header em', has_text=f'{oauth_application_name}')).to_have_count(1, timeout=transition_timeout)


await run_pw(_step)


## GakuNinRDMの「アドオンを選択」に戻り、「アドオンを選択」のパネル内「JAIRO Cloud」の行を「有効にする」をクリックする。

「JAIRO Cloudアドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    await page.goto(addon_page_url)
    await expect(page.locator(f'//h4[@class="addon-title"][normalize-space(.)="{addon_name}"]')).to_be_visible(timeout=transition_timeout)

    jairo_row = page.locator('div.addon-container[name="weko"][status="disabled"]')
    enable_link = jairo_row.get_by_text("有効にする")
    await expect(enable_link).to_be_visible(timeout=transition_timeout)
    await enable_link.click()

    terms_dialog = page.locator('.modal-dialog:has-text("JAIRO Cloud アドオン規約")')
    await expect(terms_dialog).to_be_visible(timeout=transition_timeout)
    await expect(terms_dialog.get_by_role("button", name="確認")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認」をクリックする	

「アドオンを構成」のパネル内に「JAIRO Cloud」の行が追加されること

In [ ]:
async def _step(page):
    terms_dialog = page.locator('.modal-dialog:has-text("JAIRO Cloud アドオン規約")')
    await terms_dialog.get_by_role("button", name="確認").click()

    await expect(terms_dialog).not_to_be_visible(timeout=transition_timeout)

    await expect(page.locator('#wekoScope h4.addon-title:has-text("JAIRO Cloud")')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('a[href="#wekoInputCredentials"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「アドオンを構成」のパネルの「JAIRO Cloud」の行のConnect Acconutをクリック

- Connect a JAIRO Cloud Accountダイアログが開くこと
- JAIRO Cloud Repository コンボボックスのアイテムに、No.12で設定したJAIRO Cloud application（ams-ir） が表示されること

In [ ]:
async def _step(page):
    await page.locator('a[href="#wekoInputCredentials"]').click()

    connect_dialog = page.locator('#wekoInputCredentials')
    await expect(connect_dialog).to_be_visible(timeout=transition_timeout)
    await expect(connect_dialog.get_by_role("heading", name="JAIRO Cloudアカウントに接続")).to_be_visible()

    repo_select = connect_dialog.locator('select#hostSelect')
    await expect(repo_select).to_be_visible()
    await expect(repo_select.locator(f'option:has-text("{oauth_application_name}")')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)


## JAIRO Cloud Repository コンボボックスのアイテム No.12で設定したams-irを選択して、Connectボタンをクリックする

ブラウザの別タブで、JAIRO Cloudのログイン画面が表示されること

In [ ]:
oauth_page = None

async def _step(page):
    global oauth_page
    connect_dialog = page.locator('#wekoInputCredentials')
    repo_select = connect_dialog.locator('select#hostSelect')
    await repo_select.select_option(label=f'{oauth_application_name}')

    connect_button = connect_dialog.get_by_role("button", name="接続")
    async with page.context.expect_page() as new_page_info:
        await connect_button.click()

    oauth_page = await new_page_info.value
    await oauth_page.wait_for_load_state("domcontentloaded")
    print(oauth_page.url)
    assert weko_url in oauth_page.url

await run_pw(_step)


## JAIRO Cloudのログイン画面でメールアドレスとパスワードを入力して、ログインボタンをクリックする
 
 JAIRO CloudのAuthorize applicationが表示されること

In [ ]:
async def _step(page):
    locator = page.locator('//a[contains(@class, "login-button")]')
    if await locator.is_visible():
        await oauth_page.locator('#email').fill(weko_admin_email)
        await oauth_page.locator('#password').fill(weko_admin_password)
        await oauth_page.get_by_role("button", name="Log In").click()

    await expect(oauth_page.locator("button", has_text="Authorize application")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## JAIRO CloudのAuthorize applicationで Authorize applicationをクリックする

JAIRO Cloudアカウントに接続しますか？ダイアログが表示されること  
※ダイアログが表示されずJAIRO Cloudのフロントエンドページが表示された場合は、No.18の手順に戻って繰り返してください

In [ ]:
async def _step(page):
    await oauth_page.locator("button", has_text="Authorize application").click()

    confirm_dialog = page.locator('.modal-dialog:has-text("JAIRO Cloudアカウントに接続しますか？")')
    await expect(confirm_dialog).to_be_visible(timeout=transition_timeout)
    await expect(confirm_dialog.get_by_role("button", name="接続")).to_be_visible()
    time.sleep(1)

await run_pw(_step)

## JAIRO Cloudアカウントに接続しますか？ダイアログで接続ボタンをクリックする

「アドオンを構成」のパネルの「JAIRO Cloud」の行にindex選択のコンボボックスが表示されること

In [ ]:
async def _step(page):
    confirm_dialog = page.locator('.modal-dialog:has-text("JAIRO Cloudアカウントに接続しますか？")')
    await confirm_dialog.get_by_role("button", name="接続").click()

    weko_scope = page.locator('#wekoScope')
    await expect(weko_scope.locator('.weko-settings')).to_be_visible(timeout=transition_timeout)
    index_select = weko_scope.locator( '.weko-settings select.form-control')
    await index_select.scroll_into_view_if_needed()
    await expect(index_select).to_be_visible()
    await expect(index_select.locator('option')).not_to_have_count(0)

await run_pw(_step)

## index選択のコンボボックスで「Sample Index」を選択して、saveをクリックする

プロジェクトダッシュボートのファイルリストに、JAIRO Cloudストレージが追加されること

In [ ]:
async def _step(page):
    weko_scope = page.locator('#wekoScope')
    index_select = weko_scope.locator('.weko-settings select.form-control')
    await index_select.select_option(label=f'{weko_index_name}')
    await weko_scope.get_by_role( "button", name="保存").click()
    time.sleep(1)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

ファイル画面が表示されること

In [ ]:
async def _step(page):
    await page.locator("#projectNavFiles").click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_storage_title_locator(page, 'JAIRO Cloud')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## ファイル一覧の「NII Storage」メニューの上にファイル(text1.csv)をドロップする

ファイル一覧の「NII Storage」の下のツリーにファイルが追加されること

In [ ]:
csv_file_name = 'text1.csv'
csv_file_path = os.path.join('resources/Datatest-Metadataアドオン-未病データベース', csv_file_name)
async def _step(page):
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, csv_file_path)
    time.sleep(1)

    await grdm.wait_for_uploaded(page, csv_file_name)

await run_pw(_step)

## ファイル一覧のファイル（text1.csv）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする

メタデータ入力ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, csv_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ入力ダイアログで、メタデータ様式コンボボックスから「ムーンショット目標2未病データベース-メタデータ」を選択する

メタデータ項目が切り替わること

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('ムーンショット目標2データベース（未病DB）のメタデータ登録')

    await expect(page.locator('.form-group:has-text("測定手順・条件等")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 測定手順・条件等の「項目を表示」をクリックする

項目が表示されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("測定手順・条件等")
    assert await form.is_section_expanded("測定手順・条件等")

await run_pw(_step)

## 測定手順・条件等の各項目を入力する（→以降を入力値とする）

- 入力が可能であること
- 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "測定手順・条件等"

    # 一括入力
    await form.batch_fill_fields(section, {
        "測定対象（日本語）": "血圧",
        "Object of measurement (English)": "Blood pressure",
        "関連する器官名など": "心臓",
        "データの種類（日本語）": "最大値データ"
    })

    # 値の確認
    print(await form.get(section, "測定対象（日本語）"))  # => "血圧"
    print(await form.get(section, "Object of measurement (English)"))  # => "Blood pressure"

    # 項目を非表示化
    await form.collapse_section(section)
    is_hidden = not await form.is_section_expanded(section)
    print(f"{section} is hidden: {is_hidden}")  # => True

await run_pw(_step)

## フォルダ構成の「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("フォルダ構成")
    assert await form.is_section_expanded("フォルダ構成"), "フォルダ構成 section should be expanded"

await run_pw(_step)

## フォルダ構成の各項目を入力する（→以降を入力値とする）
・フォルダの概要説明（格納されているファイルなど）（日本語） → 日付ごとに整理  
・備考（日本語） → データは毎日午前9時に収集  
・Remarks (English) → Data is collected daily at 9 AM.  
入力後に「項目を非表示」をクリックする  

- 入力が可能であること
- 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "フォルダ構成"
    
    await form.batch_fill_table_row(section, "データ格納フォルダの説明", 0, [
        "", "日付ごとに整理"
    ])
    
    await form.batch_fill_fields(section, {
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."
    })
    
    # 項目を非表示化
    await form.collapse_section(section)
    assert not await form.is_section_expanded(section), f"{section} should be hidden"

await run_pw(_step)

## テキストファイルの「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("テキストファイル")
    assert await form.is_section_expanded("テキストファイル"), "テキストファイル section should be expanded"

await run_pw(_step)

## テキストファイルの各項目を入力する（→以降を入力値とする）

- 格納先フォルダ → /data/heart_rate/
- ファイル名、または命名規則・拡張子（e.g. csv, tsv, txt）など → 命名規則: [測定対象]_[日付].csv
- 説明（日本語） → 心拍数の測定データ
- Description (English) → heart rate measurement data.
- 行の説明 →   
　　行、列の位置（日本語）→　1行目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of row (English) → Row 1  
　　Name of term (English) → Temperature  
- 列の説明 →   
　　行、列の位置（日本語） → 1列目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of column (English) → Column 1  
　　Name of term (English) → Temperature  
　　Description of term (English) → Measured temperature data (Celsius)  
- データ前処理（日本語） → 異常値は除外する。
- Data preprocessing (English) → Outliers are excluded.
- 経時的に測定したデータ → Yes
- 同種のファイル概数 → 50
- 備考（日本語） → データは毎日午前9時に収集
- Remarks (English) → Data is collected daily at 9 AM.

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "テキストファイル"
    
    # ファイル名テーブルに入力
    await form.batch_fill_table_row(section, "", 0, [
        "/data/heart_rate/",
        "命名規則: [測定対象]_[日付].csv"
    ], use_index=True)
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "説明（日本語）": "心拍数の測定データ",
        "Description (English)": "heart rate measurement data."
    })
    
    # 行の説明テーブルに入力
    await form.batch_fill_table_row(section, "行の説明", 0, [
        "1行目", "温度", "測定された温度データ（摂氏）", "Row 1", "Temperature"
    ])
    
    # 列の説明テーブルに入力
    await form.batch_fill_table_row(section, "列の説明", 0, [
        "1列目", "温度", "測定された温度データ（摂氏）", "Column 1", "Temperature", 
        "Measured temperature data (Celsius)"
    ])
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "データ前処理（日本語）": "異常値は除外する。",
        "Data preprocessing (English)": "Outliers are excluded.",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "50",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."
    })

await run_pw(_step)

## テキストファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする

① text1.csvの情報が以下の様に取得されること  
- 行数 → 101
- 列数 → 5
- 区切り文字 →  comma
- 文字コード →  ascii

② 項目が非表示となること  

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "テキストファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "行数": "101",
        "列数": "5",
        "区切り文字": "comma",
        "文字コード": "ascii"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("テキストファイル")
    assert not await form.is_section_expanded("テキストファイル"), "テキストファイル should be hidden"

await run_pw(_step)

## 任意のファイルの「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("任意のファイル")
    assert await form.is_section_expanded("任意のファイル"), "任意のファイル section should be expanded"

await run_pw(_step)

## 任意ファイルの各項目を入力する。（→以降を入力値とする）

- 格納先フォルダ ／ ファイル名、または命名規則・拡張子など  →  /data/heart_rate/
- 説明（日本語） → 心拍数の測定データ
- Description (English) → heart rate measurement data.
- データ前処理（日本語） → 異常値は除外する。
- Data Preprocessing (English) → Outliers are excluded.
- 経時的に測定したデータ → Yes
- 同種のファイル概数 → 50
- テキスト/バイナリ → Text
- ユーザー定義メタデータ項目 →  
　　メタデータ項目名（日本語） → サンプル保存条件  
　　値または内容（日本語） → Sample Storage Conditions  
　　Metadata item name(English) → -80°C  
　　value or content(English) → -80°C  
- 備考（日本語） → データは毎日午前9時に収集  
- Remarks (English) → Data is collected daily at 9 AM.	

入力が可能であること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "任意のファイル"
    
    # ファイル名テーブルに入力
    await form.batch_fill_table_row(section, "", 0, [
        "/data/heart_rate/"
    ], use_index=True)
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "説明（日本語）": "心拍数の測定データ",
        "Description (English)": "heart rate measurement data."
    })
    
    # ユーザー定義メタデータ項目テーブルに入力
    await form.batch_fill_table_row(section, "ユーザー定義メタデータ項目", 0, [
        "サンプル保存条件",
        "Sample Storage Conditions",
        "-80°C",
        "-80°C"
    ])
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "データ前処理（日本語）": "異常値は除外する。",
        "Data Preprocessing (English)": "Outliers are excluded.",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "50",
        "テキスト/バイナリ": "Text",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."
    })

await run_pw(_step)

## 任意ファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする

① text1.csvの情報が以下の様に取得されること  
- 行数 → 101
- 列数 → 5
- 区切り文字 →  comma
- 文字コード →  ascii

② 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "任意のファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "行数": "101",
        "列数": "5",
        "区切り文字": "comma",
        "文字コード": "ascii"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("任意のファイル")
    assert not await form.is_section_expanded("任意のファイル"), "任意のファイル should be hidden"

await run_pw(_step)

## 「保存」ボタンをクリックする

メタデータ入力ダイアログが閉じること

In [ ]:
async def _step(page):
    await page.locator('#treeGrid').get_by_role("link", name="保存").click()
    
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.tb-row:has-text("text1.csv")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 再度、ファイル一覧のファイル（text1.csv）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする(確認後、ダイアログは閉じる)

- メタデータ入力ダイアログが表示されること
- 入力したデータが表示されること

In [ ]:
async def _step(page):
    # ファイルを選択
    await grdm.get_select_file_extension_locator(page, csv_file_name).click()
    
    # メタデータ編集ボタンをクリック
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ編集"]').click()
    
    # メタデータ入力ダイアログが表示されること
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

    # 入力したデータが表示されることを確認
    form = FileMibyouDbMetadataForm(page)
    print("\n=== 測定手順・条件等のデータ確認 ===")
    
    # 測定手順・条件等のデータを確認
    await form.expand_section("測定手順・条件等")
    await form.batch_validate_fields("測定手順・条件等", {
        "測定対象（日本語）": "血圧",
        "Object of measurement (English)": "Blood pressure",
        "関連する器官名など": "心臓",
        "データの種類（日本語）": "最大値データ"
    })
    print("測定手順・条件等: すべてのデータが正しく保存されています")
    await form.collapse_section("測定手順・条件等")
    
    print("\n=== フォルダ構成のデータ確認 ===")
    # フォルダ構成のデータを確認
    await form.expand_section("フォルダ構成")
    await form.validate_table_row("フォルダ構成", "データ格納フォルダの説明", 0, 
                            ["", "日付ごとに整理"])
    await form.batch_validate_fields("フォルダ構成", {
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."
    })
    print("フォルダ構成: すべてのデータが正しく保存されています")
    await form.collapse_section("フォルダ構成")
    
    print("\n=== テキストファイルのデータ確認 ===")
    # テキストファイルのデータを確認
    await form.expand_section("テキストファイル")
    
    # ファイル名テーブルの確認
    await form.validate_table_row("テキストファイル", "", 0, 
                            ["/data/heart_rate/", "命名規則: [測定対象]_[日付].csv"], use_index=True)
    
    # 基本フィールドの確認
    await form.batch_validate_fields("テキストファイル", {
        "説明（日本語）": "心拍数の測定データ",
        "Description (English)": "heart rate measurement data.",
        "データ前処理（日本語）": "異常値は除外する。",
        "Data preprocessing (English)": "Outliers are excluded.",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "50",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM.",
        "行数": "101",
        "列数": "5",
        "区切り文字": "comma",
        "文字コード": "ascii"
    })
    
    # 行の説明テーブルの確認
    await form.validate_table_row("テキストファイル", "行の説明", 0,
                            ["1行目", "温度", "測定された温度データ（摂氏）", "Row 1", "Temperature"])
    
    # 列の説明テーブルの確認
    await form.validate_table_row("テキストファイル", "列の説明", 0,
                            ["1列目", "温度", "測定された温度データ（摂氏）", "Column 1", "Temperature", 
                             "Measured temperature data (Celsius)"])
    
    print("テキストファイル: すべてのデータが正しく保存されています")
    await form.collapse_section("テキストファイル")
    
    print("\n=== 任意のファイルのデータ確認 ===")
    # 任意のファイルのデータを確認
    await form.expand_section("任意のファイル")
    
    # ファイル名テーブルの確認
    await form.validate_table_row("任意のファイル", "", 0, 
                            ["/data/heart_rate/"], use_index=True)
    
    # 基本フィールドの確認
    await form.batch_validate_fields("任意のファイル", {
        "説明（日本語）": "心拍数の測定データ",
        "Description (English)": "heart rate measurement data.",
        "データ前処理（日本語）": "異常値は除外する。",
        "Data Preprocessing (English)": "Outliers are excluded.",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "50",
        "テキスト/バイナリ": "Text",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM.",
        "行数": "101",
        "列数": "5",
        "区切り文字": "comma",
        "文字コード": "ascii"
    })
    
    # ユーザー定義メタデータ項目テーブルの確認
    await form.validate_table_row("任意のファイル", "ユーザー定義メタデータ項目", 0,
                            ["サンプル保存条件", "Sample Storage Conditions", "-80°C", "-80°C"])
    
    print("任意のファイル: すべてのデータが正しく保存されています")
    await form.collapse_section("任意のファイル")
    
    print("\n=== すべてのメタデータが正しく保存され、再表示されました ===")
    
    # ダイアログを閉じる
    await page.locator('#treeGrid').get_by_role("link", name="閉じる").click()
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    print("メタデータダイアログを閉じました")

await run_pw(_step)

## ファイル一覧の「NII Storage」メニューの上にtext1.tsvをドロップする

ファイル一覧の「NII Storage」の下のツリーにファイルが追加されること

In [ ]:
tsv_file_name = 'text1.tsv'
tsv_file_path = os.path.join('resources/Datatest-Metadataアドオン-未病データベース', tsv_file_name)
async def _step(page):
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, tsv_file_path)
    time.sleep(1)

    await grdm.wait_for_uploaded(page, tsv_file_name)

await run_pw(_step)

## ファイル一覧のファイル（text1.tsv）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする	

メタデータ入力ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, tsv_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ入力ダイアログで、メタデータ様式コンボボックスから「ムーンショット目標2未病データベース-メタデータ」を選択する

メタデータ項目が切り替わること

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('ムーンショット目標2データベース（未病DB）のメタデータ登録')

    await expect(page.locator('.form-group:has-text("測定手順・条件等")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## テキストファイルの「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("テキストファイル")
    assert await form.is_section_expanded("テキストファイル"), "テキストファイル section should be expanded"

await run_pw(_step)

## 任意ファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする
① text1.tsvの情報が以下の様に取得されること
- 行数 → 101
- 列数 → 5
- 区切り文字 →  tab
- 文字コード →  ascii

② 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "テキストファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "行数": "101",
        "列数": "5",
        "区切り文字": "tab",
        "文字コード": "ascii"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("テキストファイル")
    assert not await form.is_section_expanded("テキストファイル"), "テキストファイル should be hidden"

await run_pw(_step)

## 「保存」ボタンをクリックする

メタデータ入力ダイアログが閉じること

In [ ]:
async def _step(page):
    await page.locator('#treeGrid').get_by_role("link", name="保存").click()
    
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.tb-row:has-text("text1.tsv")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の「NII Storage」メニューの上にexcel1.xlsxをドロップする

ファイル一覧の「NII Storage」の下のツリーにファイルが追加されること

In [ ]:
excel_file_name = 'excel1.xlsx'
excel_file_path = os.path.join('resources/Datatest-Metadataアドオン-未病データベース', excel_file_name)
async def _step(page):
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, excel_file_path)
    time.sleep(1)

    await grdm.wait_for_uploaded(page, excel_file_name)

await run_pw(_step)

## ファイル一覧のファイル（excel1.xlsx）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする

メタデータ入力ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, excel_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ入力ダイアログで、メタデータ様式コンボボックスから「ムーンショット目標2未病データベース-メタデータ」を選択する

メタデータ項目が切り替わること

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('ムーンショット目標2データベース（未病DB）のメタデータ登録')

    await expect(page.locator('.form-group:has-text("測定手順・条件等")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## エクセルファイルの「項目を表示」をクリックする

項目が表示されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("エクセルファイル")
    assert await form.is_section_expanded("エクセルファイル"), "エクセルファイル section should be expanded"

await run_pw(_step)

## エクセルファイルの各項目を入力する（→以降を入力値とする）
- 格納先フォルダ →  /data/heart_rate/
- ファイル名、または命名規則・拡張子（e.g. xlsx, xlsm）など →  命名規則: [測定対象]_[日付].csv
- 説明（日本語） → 心拍数の測定データ
- Description (English) → heart rate measurement data.
- 行の説明 →   
　　行の位置（日本語）→　1行目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of row (English) → Row 1  
　　Name of term (English) → Temperature  
　　Description of term (English) → Measured temperature data (Celsius)   
- 列の説明 →   
　　列の位置（日本語） → 1列目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of column (English) → Column 1  
　　Name of term (English) → Temperature  
　　Description of term (English) → Measured temperature data (Celsius)  
- データ前処理（日本語） → 異常値は除外する。
- Data Preprocessing (English) → Outliers are excluded.
- 経時的に測定したデータ → Yes
- 同種のファイル概数 → 10
- 備考（日本語） → データは毎日午前9時に収集
- Remarks (English) → Data is collected daily at 9 AM.

入力が可能であること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "エクセルファイル"
    
    # ファイル名テーブルに入力
    await form.batch_fill_table_row(section, "", 0, [
        "/data/heart_rate/",
        "命名規則: [測定対象]_[日付].csv"
    ], use_index=True)
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "説明（日本語）": "心拍数の測定データ",
        "Description (English)": "heart rate measurement data."
    })
    
    # 行の説明テーブルに入力
    await form.batch_fill_table_row(section, "行の説明", 0, [
        "1行目", "温度", "測定された温度データ（摂氏）", "Row 1", "Temperature",
        "Measured temperature data (Celsius)"
    ])
    
    # 列の説明テーブルに入力
    await form.batch_fill_table_row(section, "列の説明", 0, [
        "1列目", "温度", "測定された温度データ（摂氏）", "Column 1", "Temperature",
        "Measured temperature data (Celsius)"
    ])
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "データ前処理（日本語）": "異常値は除外する。",
        "Data Preprocessing (English)": "Outliers are excluded.",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "10",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."
    })

await run_pw(_step)

## エクセルファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする

① excel1.xlsxの情報が以下の様に取得されること
- 行数 → 51
- 列数 → 5

② 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "エクセルファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "行数": "51",
        "列数": "5"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("エクセルファイル")
    assert not await form.is_section_expanded("エクセルファイル"), "エクセルファイル should be hidden"

await run_pw(_step)

## 「保存」ボタンをクリックする

メタデータ入力ダイアログが閉じること

In [ ]:
async def _step(page):
    await page.locator('#treeGrid').get_by_role("link", name="保存").click()
    
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.tb-row:has-text("excel1.xlsx")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 再度、ファイル一覧のファイル（excel1.xlsx）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする(確認後、ダイアログは閉じる)

- メタデータ入力ダイアログが表示されること
- 入力したデータが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, excel_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)
    
    # 入力したデータが表示されることを確認
    print("\n=== エクセルファイルのデータ確認 ===")
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("エクセルファイル")
    
    # ファイル名テーブルの確認
    await form.validate_table_row("エクセルファイル", "", 0,
                            ["/data/heart_rate/", "命名規則: [測定対象]_[日付].csv"], use_index=True)
    
    # 基本フィールドの確認
    await form.batch_validate_fields("エクセルファイル", {
        "説明（日本語）": "心拍数の測定データ",
        "Description (English)": "heart rate measurement data.",
        "データ前処理（日本語）": "異常値は除外する。",
        "Data Preprocessing (English)": "Outliers are excluded.",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "10",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM.",
        "行数": "51",
        "列数": "5"
    })
    
    # 行の説明テーブルの確認
    await form.validate_table_row("エクセルファイル", "行の説明", 0,
                            ["1行目", "温度", "測定された温度データ（摂氏）", "Row 1", "Temperature",
                             "Measured temperature data (Celsius)"])
    
    # 列の説明テーブルの確認
    await form.validate_table_row("エクセルファイル", "列の説明", 0,
                            ["1列目", "温度", "測定された温度データ（摂氏）", "Column 1", "Temperature",
                             "Measured temperature data (Celsius)"])
    
    print("エクセルファイル: すべてのデータが正しく保存されています")
    await form.collapse_section("エクセルファイル")
    
    # ダイアログを閉じる
    await page.locator('#treeGrid').get_by_role("link", name="閉じる").click()
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    print("メタデータダイアログを閉じました")

await run_pw(_step)

## ファイル一覧の「NII Storage」メニューの上にimage1.jpegをドロップする

ファイル一覧の「NII Storage」の下のツリーにファイルが追加されること

In [ ]:
image_file_name = 'image1.jpeg'
image_file_path = os.path.join('resources/Datatest-Metadataアドオン-未病データベース', image_file_name)
async def _step(page):
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, image_file_path)
    time.sleep(1)

    await grdm.wait_for_uploaded(page, image_file_name)

await run_pw(_step)

## ファイル一覧のファイル（image1.jpeg）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする

メタデータ入力ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, image_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ入力ダイアログで、メタデータ様式コンボボックスから「ムーンショット目標2未病データベース-メタデータ」を選択する

メタデータ項目が切り替わること

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('ムーンショット目標2データベース（未病DB）のメタデータ登録')

    await expect(page.locator('.form-group:has-text("測定手順・条件等")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画像ファイルの「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("画像ファイル")
    assert await form.is_section_expanded("画像ファイル"), "画像ファイル section should be expanded"

await run_pw(_step)

## 画像ファイルの各項目を入力する（→以降を入力値とする）

- 格納先フォルダ → /data/heart_rate/
- ファイル名、または命名規則・拡張子（e.g. jpg, png, tif）など → 命名規則: [測定対象]_[日付].jpeg	
- 説明（日本語） → 実験Bの顕微鏡画像データ
- Description (English) → Microscope image data from Experiment B.
- データ前処理（日本語） → ノイズ除去、コントラスト調整
- Data preprocessing (English) → Noise removal, contrast adjustment
- 経時的に測定したデータ → YES
- 色ビット数（色深度） → 24ビット
- 圧縮形式 →非圧縮
- 同種のファイル概数 → 50
- 備考（日本語） → データは毎日午前9時に収集
- Remarks (English) →Data is collected daily at 9 AM.

入力が可能であること


In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "画像ファイル"
    
    # ファイル名テーブルに入力
    await form.batch_fill_table_row(section, "", 0, [
        "/data/heart_rate/",
        "命名規則: [測定対象]_[日付].jpeg"
    ], use_index=True)
    
    # その他のフィールドを一括入力
    await form.batch_fill_fields(section, {
        "説明（日本語）": "実験Bの顕微鏡画像データ",
        "Description (English)": "Microscope image data from Experiment B.",
        "データ前処理（日本語）": "ノイズ除去、コントラスト調整",
        "Data preprocessing (English)": "Noise removal, contrast adjustment",
        "経時的に測定したデータ": "Yes",
        "色ビット数（色深度）": "24ビット",
        "圧縮形式": "非圧縮",
        "同種のファイル概数": "50",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."})
    
await run_pw(_step)

## 画像ファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする。

① image1.jpegの情報が以下の様に取得されること
- 幅ピクセル数 → 900 x 600
- 高さピクセル数 → 900 x 600 
- 解像度（水平方向） → 72dpi
- 解像度（垂直方向） → 72dpi 
- 色情報の数 → color
- 画像タイプ → jpeg

② 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "画像ファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "幅ピクセル数": "900 x 600",
        "高さピクセル数": "900 x 600",
        "解像度（水平方向）": "72 dpi",
        "解像度（垂直方向）": "72 dpi",
        "色情報の数": "color",
        "画像タイプ": "jpeg"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("画像ファイル")
    assert not await form.is_section_expanded("画像ファイル"), "画像ファイル should be hidden"

await run_pw(_step)

## 任意のファイルの「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("任意のファイル")
    assert await form.is_section_expanded("任意のファイル"), "任意のファイル section should be expanded"

await run_pw(_step)

## 任意ファイルの各項目を入力する（→以降を入力値とする）

- 格納先フォルダ → /data/heart_rate/
- ファイル名、または命名規則・拡張子など → 命名規則: [測定対象]_[日付].jpeg
- 説明（日本語） →  実験Bの顕微鏡画像データ
- Description (English) → Microscope image data from Experiment B.
- データ前処理（日本語） → ノイズ除去、コントラスト調整
- Data Preprocessing (English) → Noise removal, contrast adjustment
- 経時的に測定したデータ → Yes
- 同種のファイル概数 → 50
- テキスト/バイナリ → Binary
- ユーザー定義メタデータ項目 →   
　　メタデータ項目名（日本語） → サンプル保存条件  
　　値または内容（日本語） → Sample Storage Conditions  
　　Metadata item name (English)→ -80°C  
　　value or content (English)→ -80°C    
- 備考（日本語） →  データは毎日午前9時に収集
- Remarks (English) → Data is collected daily at 9 AM.

入力が可能であること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "任意のファイル"
    
    # ファイル名テーブルに入力
    await form.batch_fill_table_row(section, "", 0, [
        "/data/heart_rate/",
        "命名規則: [測定対象]_[日付].jpeg"
    ], use_index=True)
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "説明（日本語）": "実験Bの顕微鏡画像データ",
        "Description (English)": "Microscope image data from Experiment B."
    })
    
    # ユーザー定義メタデータ項目テーブルに入力
    await form.batch_fill_table_row(section, "ユーザー定義メタデータ項目", 0, [
        "サンプル保存条件",
        "Sample Storage Conditions",
        "-80°C",
        "-80°C"
    ])
    
    # フィールドを一括入力
    await form.batch_fill_fields(section, {
        "データ前処理（日本語）": "ノイズ除去、コントラスト調整",
        "Data Preprocessing (English)": "Noise removal, contrast adjustment",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "50",
        "テキスト/バイナリ": "Binary",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM."
    })

await run_pw(_step)

## 任意ファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする

① image1.jpegの情報が以下の様に取得されること
- 幅ピクセル数 → 900 x 600
- 高さピクセル数 → 900 x 600 
- 解像度（水平方向） → 72dpi
- 解像度（垂直方向） → 72dpi 
- 色情報の数 → color
- 画像タイプ → jpeg

② 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "任意のファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "幅ピクセル数": "900 x 600",
        "高さピクセル数": "900 x 600",
        "解像度（水平方向）": "72 dpi",
        "解像度（垂直方向）": "72 dpi",
        "色情報の数": "color",
        "画像タイプ": "jpeg"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("任意のファイル")
    assert not await form.is_section_expanded("任意のファイル"), "任意のファイル should be hidden"

await run_pw(_step)

## 「保存」ボタンをクリックする

メタデータ入力ダイアログが閉じること

In [ ]:
async def _step(page):
    await page.locator('#treeGrid').get_by_role("link", name="保存").click()
    
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.tb-row:has-text("image1.jpeg")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 再度、ファイル一覧のファイル（image1.jpeg）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする(確認後、ダイアログは閉じる)

- メタデータ入力ダイアログが表示されること
- 入力したデータが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, image_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)
    
    # 入力したデータが表示されることを確認
    print("\n=== 画像ファイルのデータ確認 ===")
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("画像ファイル")
    
    # ファイル名テーブルの確認
    await form.validate_table_row("画像ファイル", "", 0,
                            ["/data/heart_rate/", "命名規則: [測定対象]_[日付].jpeg"], use_index=True)
    
    # 基本フィールドの確認
    await form.batch_validate_fields("画像ファイル", {
        "説明（日本語）": "実験Bの顕微鏡画像データ",
        "Description (English)": "Microscope image data from Experiment B.",
        "データ前処理（日本語）": "ノイズ除去、コントラスト調整",
        "Data preprocessing (English)": "Noise removal, contrast adjustment",
        "経時的に測定したデータ": "Yes",
        "色ビット数（色深度）": "24ビット",
        "圧縮形式": "非圧縮",
        "同種のファイル概数": "50",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM.",
        "幅ピクセル数": "900 x 600",
        "高さピクセル数": "900 x 600",
        "解像度（水平方向）": "72 dpi",
        "解像度（垂直方向）": "72 dpi",
        "色情報の数": "color",
        "画像タイプ": "jpeg"
    })
    
    print("画像ファイル: すべてのデータが正しく保存されています")
    await form.collapse_section("画像ファイル")
    
    print("\n=== 任意のファイルのデータ確認 ===")
    await form.expand_section("任意のファイル")
    
    # ファイル名テーブルの確認
    await form.validate_table_row("任意のファイル", "", 0,
                            ["/data/heart_rate/", "命名規則: [測定対象]_[日付].jpeg"], use_index=True)
    
    # 基本フィールドの確認
    await form.batch_validate_fields("任意のファイル", {
        "説明（日本語）": "実験Bの顕微鏡画像データ",
        "Description (English)": "Microscope image data from Experiment B.",
        "データ前処理（日本語）": "ノイズ除去、コントラスト調整",
        "Data Preprocessing (English)": "Noise removal, contrast adjustment",
        "経時的に測定したデータ": "Yes",
        "同種のファイル概数": "50",
        "テキスト/バイナリ": "Binary",
        "備考（日本語）": "データは毎日午前9時に収集",
        "Remarks (English)": "Data is collected daily at 9 AM.",
        "幅ピクセル数": "900 x 600",
        "高さピクセル数": "900 x 600",
        "解像度（水平方向）": "72 dpi",
        "解像度（垂直方向）": "72 dpi",
        "色情報の数": "color",
        "画像タイプ": "jpeg"
    })
    
    # ユーザー定義メタデータ項目の確認
    await form.validate_table_row("任意のファイル", "ユーザー定義メタデータ項目", 0,
                            ["サンプル保存条件", "Sample Storage Conditions", "-80°C", "-80°C"])
    
    print("任意のファイル: すべてのデータが正しく保存されています")
    await form.collapse_section("任意のファイル")
    
    print("\n=== すべてのメタデータが正しく保存され、再表示されました ===")
    
    # ダイアログを閉じる
    await page.locator('#treeGrid').get_by_role("link", name="閉じる").click()
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    print("メタデータダイアログを閉じました")

await run_pw(_step)

## ファイル一覧の「NII Storage」メニューの上にimage1.pngをドロップする

ファイル一覧の「NII Storage」の下のツリーにファイルが追加されること

In [ ]:
png_file_name = 'image1.png'
png_file_path = os.path.join('resources/Datatest-Metadataアドオン-未病データベース', png_file_name)
async def _step(page):
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, png_file_path)
    time.sleep(1)

    await grdm.wait_for_uploaded(page, png_file_name)

await run_pw(_step)

## ファイル一覧のファイル（image1.png）を選択して、ツールバーの「メタデータ編集」ボタンをクリックする

メタデータ入力ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, png_file_name).click()

    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//*[text() = "メタデータ編集"]').click()

    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ入力ダイアログで、メタデータ様式コンボボックスから「ムーンショット目標2未病データベース-メタデータ」を選択する

メタデータ項目が切り替わること

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('ムーンショット目標2データベース（未病DB）のメタデータ登録')

    await expect(page.locator('.form-group:has-text("測定手順・条件等")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画像ファイルの「項目を表示」をクリックする

項目が展開されること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    await form.expand_section("画像ファイル")
    assert await form.is_section_expanded("画像ファイル"), "画像ファイル section should be expanded"

await run_pw(_step)

## 画像ファイルの下記項目の自動取得ボタンをクリックする。入力後に「項目を非表示」をクリックする。その後、保存ボタンをクリックする

① image1.pngの情報が以下の様に取得されること
- 幅ピクセル数 → 836 x 557
- 高さピクセル数 → 836 x 557 
- 解像度（水平方向） → 96.012dpi
- 解像度（垂直方向） → 96.012dpi 
- 色情報の数 → color
- 画像タイプ → png

② 項目が非表示となること

In [ ]:
async def _step(page):
    form = FileMibyouDbMetadataForm(page)
    section = "画像ファイル"
    
    # 自動取得と値の検証を統合関数で実行
    field_expectations = {
        "幅ピクセル数": "836 x 557",
        "高さピクセル数": "836 x 557",
        "解像度（水平方向）": "96.012 dpi",
        "解像度（垂直方向）": "96.012 dpi",
        "色情報の数": "color",
        "画像タイプ": "png"
    }
    
    await form.auto_fetch_and_validate(section, field_expectations)
    
    # 項目を非表示
    await form.collapse_section("画像ファイル")
    assert not await form.is_section_expanded("画像ファイル"), "画像ファイル should be hidden"
    
    # 保存ボタンをクリック
    await page.locator('#treeGrid').get_by_role("link", name="保存").click()
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.tb-row:has-text("image1.png")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

プロジェクトメタデータ参照の画面が表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトメタデータ参照の画面の「新規メタデータを作成」ボタンをクリックする

メタデータ様式の選択を求めるダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-metadata-button]').click()

    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ様式の選択を求めるダイアログで「ムーンショット目標2未病データベース（未病DB）のメタデータ登録 」のラジオボタンを選択して「メタデータを作成」をクリックする

- メタデータの各項目を入力する画面が表示されること
- タイトル下に注釈が表示されること（文言は下記）

 本登録画面ではムーンショット目標2データベース（未病DB）の【プロジェクトメタデータ】を登録することができます。研究プロジェクトの概要情報やデータセットの管理情報を記載できます。

なお、未病DBではデータセット全体の管理情報に対するメタデータを「プロジェクトメタデータ」、実験やデータの詳細に対するメタデータを「データセットメタデータ」、個々のファイルやフォルダに対するメタデータを「ファイルメタデータ」、その他追加のメタデータファイルを「拡張メタデータ」としています。各メタデータについて詳しくは未病DBガイドラインを参照ください。
未病DBガイドライン： https://rdm.example.com/3sw6v1/  


In [ ]:
async def _step(page):
    await page.get_by_text( "ムーンショット目標2データベース（未病DB）のメタデータ登録").click()
    await page.locator('//*[@data-test-new-report-modal-create-report-button]').click()

    await expect( page.locator('[data-test-page-heading]', has_text='プロジェクトメタデータ登録')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## メタデータの各項目を入力する画面で「未病データベース_プロジェクトメタデータ」シートの記入例を元に各項目に入力する（但し、「謝辞に記載する名称（個人名、グループ名等）、または謝辞全文」は未入力とする）、入力後に画面左側ナビゲーターから「ファイルメタデータ」をクリックする

- 入力が可能であること
- ナビゲータクリックでファイルメタデータ画面に遷移すること

In [ ]:
async def select_by_label(page, label_text, option_text):
    trigger = page.locator(
        f'//label[.//p[contains(normalize-space(), "{label_text}")]]'
    ).locator(
        'xpath=following-sibling::div//div[contains(@class,"ember-power-select-trigger")]'
    ).first

    await trigger.scroll_into_view_if_needed()
    await trigger.click()

    option = page.locator(
        f'//div[contains(@id,"ember-basic-dropdown-wormhole")]'
        f'//li[@role="option"][normalize-space()="{option_text}"]'
    )

    await option.wait_for(state="visible")
    await option.click() 


async def _step(page):
    await expect(page.locator('h2:has-text("プロジェクトメタデータ登録")')).to_be_visible(timeout=transition_timeout)

    await expect(page.locator('input[name="__responseKey_title-of-dataset"]')).to_have_value(f'{rdm_project_name}', timeout=transition_timeout)
    await page.locator('//input[@name="__responseKey_title-of-dataset-en"]/following-sibling::button[text()="自動取得"]').click();
    await expect(page.locator('input[name="__responseKey_title-of-dataset-en"]')).to_have_value(f'{rdm_project_name}', timeout=transition_timeout)
    await select_by_label(page, 'MSプロジェクト名 [必須]', 'MS2合原PJ')
    await page.locator('input[name="__responseKey_data-id"]').fill('ms2dataid')
    
    today = datetime.now().strftime('%Y-%m-%d')
    await expect(page.locator('input[name="__responseKey_date-registered-in-metadata"]')).to_have_value(f'{today}', timeout=transition_timeout)
    await expect(page.locator('input[name="__responseKey_date-updated-in-metadata"]')).to_have_value(f'{today}', timeout=transition_timeout)

    await page.locator('input[name="__responseKey_purpose-of-experiment"]').fill('細胞成長のメカニズムを解明するため')
    await page.locator('input[name="__responseKey_purpose-of-experiment-en"]').fill('To elucidate the mechanisms of cell growth')
    await page.locator('input[name="__responseKey_description-of-experimental-condition"]').fill('実験は2025年3月1日から3月31日まで行われ、顕微鏡と温度計を使用して細胞の成長を観察しました。')
    await page.locator('input[name="__responseKey_description-of-experimental-condition-en"]').fill('The experiment was conducted from March 1 to March 31, 2025, using a microscope and thermometer to observe cell growth.')
    
    await page.locator('h3:has-text("キーワード") ~ div button:has-text("アイテム追加")').first.click()
    await page.locator('input[name="__responseKey_keywords|filename"]').fill('細胞成長')
    await page.locator('input[name="__responseKey_keywords|filename-en"]').fill('Cell growth')
    
    await select_by_label(page, 'データセットの分野 [必須]', 'ライフサイエンス')

    analysis_checkboxes = page.locator('input[name="__responseKey_Analysis-type"]')
    checkbox_count = await analysis_checkboxes.count()
    for i in range(checkbox_count):
        if not await analysis_checkboxes.nth(i).is_checked():
            await analysis_checkboxes.nth(i).check()

    await page.locator('textarea[name="__responseKey_Analysis-type-other"]').fill('フェノームデータ')
    await select_by_label(page, '他のデータベースで特定のモダリティ用に作成したメタデータファイルの有無 [必須]', '有')

    await page.locator('h3:has-text("（有りの場合）メタデータファイル名") ~ div button:has-text("アイテム追加")').first.click()
    await page.locator('input[name="__responseKey_metadata-filename|filename"]').fill('フェノームデータ_メタデータ_20250321.txt')
    
    await select_by_label(page, '連絡・許諾の要不要 [必須]', '連絡不要')
    await select_by_label(page, '謝辞に記載の要不要 [必須]', '不要')
    await page.locator('input[name="__responseKey_names-to-be-included-in-the-acknowledgments-en"]').fill('Taro Tanaka、Hanako Yamada')
    await page.locator('input[name="__responseKey_other-conditions-or-special-notes"]').fill('本データは特定のソフトウェアバージョンでのみ解析可能です。')
    await page.locator('input[name="__responseKey_other-conditions-or-special-notes-en"]').fill('This data can only be analyzed with a specific software version.')

    await select_by_label(page, 'ライセンス [必須]', 'ライセンスなし')
    await select_by_label(page, '有償・無償 [必須]', '無償')
    await select_by_label(page, '商用利用の可否 [必須]', '否')
    await select_by_label(page, 'アクセス権 [必須]', '共有')

    await page.locator('input[name="__responseKey_scheduled-release-date"]').fill(scheduled_release_date)
    await select_by_label(page, 'リポジトリ情報 [必須]', 'GakuNin RDM | GakuNin RDM')
    await page.locator('input[name="__responseKey_repository-url-doi-link"]').fill('https://doi.org/10.1234/example1')
    await page.locator('textarea[name="__responseKey_other-supplementary-information"]').fill('データは特定の条件下でのみ使用可能です。')
    await page.locator('textarea[name="__responseKey_other-supplementary-information-en"]').fill('Data can only be used under specific conditions.')
        
    await page.locator('h3:has-text("データ作成者") ~ div button:has-text("アイテム追加")').first.click()
    await page.locator('input[name="__responseKey_data-creator|name"]').fill('山本一郎')
    await page.locator('input[name="__responseKey_data-creator|name-en"]').fill('Ichiro Yamamoto')
    await page.locator('input[name="__responseKey_data-creator|belonging"]').fill('東西大学 生物学部')
    await page.locator('input[name="__responseKey_data-creator|belonging-en"]').fill('Department of Biology, Tozai University')
    await page.locator('input[name="__responseKey_data-creator|contact"]').fill('ichiro.yamamoto@example.com')
    
    await page.locator('h3:has-text("データ管理者") ~ div button:has-text("アイテム追加")').first.click()
    await page.locator('input[name="__responseKey_data-manager|name"]').fill('鈴木花子')
    await page.locator('input[name="__responseKey_data-manager|name-en"]').fill('Hanako Suzuki')
    await page.locator('input[name="__responseKey_data-manager|belonging"]').fill('南北大学 化学部')
    await page.locator('input[name="__responseKey_data-manager|belonging-en"]').fill('Department of Chemistry, Nanboku University')
    await page.locator('input[name="__responseKey_data-manager|contact"]').fill('hanako.suzuki@example.com')
    
    await page.locator('input[name="__responseKey_target-type-of-acquired-data"]').fill('ヒト細胞')
    await page.locator('input[name="__responseKey_target-type-of-acquired-data-en"]').fill('Human cells')
    await page.locator('input[name="__responseKey_ethics-review-committee-approval"]').fill('承認済み（2025年2月15日）')
    await page.locator('input[name="__responseKey_ethics-review-committee-approval-en"]').fill('Approved (February 15, 2025)')
    
    await select_by_label(page, '（ヒト）インフォームドコンセント（IC） 有・無・不要 [必須]', '有')
    await select_by_label(page, '（IC有の場合）第三者提供の同意', '無')
    await select_by_label(page, '（IC有の場合）海外提供', '有')
    await select_by_label(page, '（IC有の場合）産業利用等', '無')
    await select_by_label(page, '（IC無の場合', '同意不要')
    await select_by_label(page, '（ヒト）匿名加工の有無', '有')
    await select_by_label(page, '利益相反の有無 [必須]', '有')
    
    await page.locator('input[name="__responseKey_conflict-of-interest"]').fill('XYZ株式会社')
    await page.locator('input[name="__responseKey_conflict-of-interest-en"]').fill('Company "XYZ Corporation"')

    await page.locator('textarea[name="__responseKey_remarks-3"]').fill('利益相反の詳細は、プロジェクトの資金提供者が実験結果に影響を与える可能性があることです。')
    await page.locator('textarea[name="__responseKey_remarks-3-en"]').fill('Details of the conflict of interest include the possibility that the project funder, the fictional company "ABC Corporation," may influence the experimental results.')
    
    await page.get_by_role("link", name="ファイルメタデータ").click()
    await expect(page.locator('h2:has-text("ファイルメタデータ")')).to_be_visible(timeout=transition_timeout)


await run_pw(_step)

## ファイルメタデータのツリーにファイル（text1.csv）の左端のチェックボックスをチェックして、「内容確認」ボタンをクリックする

- 内容確認画面が表示されること
- 「謝辞に記載する名称（個人名、グループ名等）、または謝辞全文」に警告「このフィールドをブランクにはできません。」が表示されていること
- ページの最下に移動して「(MS2共有) カタログ登録前チェック日付」に編集アイコンが表示されていなこと
- 「（MS2共有）カタログ登録前チェックリストバージョン」に編集アイコンが表示されていないこと


In [ ]:
async def _step(page):
    await page.locator('//tr[.//p[normalize-space()="text1.csv"]]''//button[@data-test-file-metadata-input-upper]').click()
    await page.get_by_role("link", name="内容確認").click()

    await expect(page.locator('h2:has-text("プロジェクトメタデータ登録")')).to_be_visible(timeout=transition_timeout)

    acknowledgments_locator = page.locator('[data-test-validation-errors="__responseKey_names-to-be-included-in-the-acknowledgments"]')
    await acknowledgments_locator.scroll_into_view_if_needed()
    await expect(acknowledgments_locator).to_be_visible()
    await expect(acknowledgments_locator).to_have_text("このフィールドをブランクにはできません。")

    await page.locator('//label[.//p[contains(normalize-space(),"（MS2共有）カタログ登録前チェック日付")]]').scroll_into_view_if_needed()
    edit_icon = page.locator('//label[.//p[contains(normalize-space(),"（MS2共有）カタログ登録前チェック日付")]]''//i[contains(@class,"fa-edit")]')
    await expect(edit_icon).to_have_count(0)
    edit_icon = page.locator('//label[.//p[contains(normalize-space(),"（（MS2共有）カタログ登録前チェックリストバージョン")]]''//i[contains(@class,"fa-edit")]')
    await expect(edit_icon).to_have_count(0)

await run_pw(_step)

## 「謝辞に記載する名称（個人名、グループ名等）、または謝辞全文」の編集ボタンをクリックする

メタデータの各項目を入力する画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//label[.//p[contains(normalize-space(),"謝辞に記載する名称（個人名、グループ名等）、または謝辞全文（日本語） [必須]")]]''//i[contains(@class,"fa-edit")]').click()
    await expect(page.locator('h2:has-text("プロジェクトメタデータ登録")')).to_be_visible(timeout=transition_timeout)

    acknowledgments_locator = page.locator('[data-test-validation-errors="__responseKey_names-to-be-included-in-the-acknowledgments"]')
    await expect(acknowledgments_locator).to_be_visible()
    await expect(acknowledgments_locator).to_have_text("このフィールドをブランクにはできません。")

await run_pw(_step)

## メタデータの各項目を入力する画面で「謝辞に記載する名称（個人名、グループ名等）、または謝辞全文」に”TEST”を入力し、ナビゲーターの「データセットメタデータ登録」をクリックする

データセットメタデータ登録画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('input[name="__responseKey_names-to-be-included-in-the-acknowledgments"]').fill('TEST')
    await page.get_by_role("link", name="データセットメタデータ登録").click()
    await expect(page.locator('h2:has-text("データセットメタデータ登録")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## データセットメタデータ登録画面で、各項目を入力する（→以降を入力値とする）

- 測定対象（日本語）→ 血圧
- Object of measurement (English) →　Blood pressure
- 関連する器官名など → 心臓
- データの種類（日本語） → 最大値データ
- Data type（English） → Maximum value data

In [ ]:
async def _step(page):
    # 測定対象（日本語）
    await page.locator('input[name="__responseKey_d-msr-object-of-measurement-jp"]').fill('血圧')
    # Object of measurement (English)
    await page.locator('input[name="__responseKey_d-msr-object-of-measurement-en"]').fill('Blood pressure')
    
    # 関連する器官名など
    await page.locator('input[name="__responseKey_d-msr-target-organs-for-measurement"]').fill('心臓')
    
    # データの種類（日本語）
    await page.locator('input[name="__responseKey_d-msr-data-type-jp"]').fill('最大値データ')
    # Data type（English）
    await page.locator('input[name="__responseKey_d-msr-data-type-en"]').fill('Maximum value data')

await run_pw(_step)

## データ格納フォルダの説明のアイテム追加ボタンをクリックして、各項目を入力する

- フォルダ名（命名規則など） → フォルダ名: 研究データ/2025/実験C
- フォルダの概要説明（格納されているファイルなど）（日本語） → 実験Cのデータを格納したフォルダ。
- Description of folder (Contained folders and files) (English) → Folder containing data from Experiment C.
- コンテンツ（代表的な拡張子）のテストデータ → 圧力データ: pressure_20250321.csv

In [ ]:
async def _step(page):
    # データ格納フォルダの説明のアイテム追加ボタンをクリック
    await page.locator('h3:has-text("データ格納フォルダの説明") ~ div button:has-text("アイテム追加")').first.click()
    time.sleep(1)
    
    # フォルダ名（命名規則など）
    await page.locator('textarea[name="__responseKey_d-fol-Structure-or-descriptions-of-folders-jp|folder-name"]').fill('フォルダ名: 研究データ/2025/実験C')
    
    # フォルダの概要説明（格納されているファイルなど）（日本語）
    await page.locator('textarea[name="__responseKey_d-fol-Structure-or-descriptions-of-folders-jp|description-of-folder"]').fill('実験Cのデータを格納したフォルダ。')
    # Description of folder (Contained folders and files) (English)
    await page.locator('textarea[name="__responseKey_d-fol-Structure-or-descriptions-of-folders-jp|description-of-folder-en"]').fill('Folder containing data from Experiment C.')
    
    # コンテンツ（代表的な拡張子）
    await page.locator('textarea[name="__responseKey_d-fol-Structure-or-descriptions-of-folders-jp|contents"]').fill('圧力データ: pressure_20250321.csv')

await run_pw(_step)

## テキストファイルのテキストファイルアイテム追加ボタンをクリックして、各項目を入力する（→以降を入力値とする）

- ファイル名、または命名規則・拡張子（e.g. csv, tsv, txt）など → 命名規則: [測定対象]_[日付].xlsx
- 説明（日本語） → 心拍数の測定データ
- Description (English) → heart rate measurement data.
- 行の説明 →   
　　行の位置（日本語）→　1行目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of row (English) → Row 1  
　　Name of term (English) → Temperature  
- 列の説明 →   
　　行の位置（日本語） → 1列目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of column (English) → Column 1  
　　Name of term (English) → Temperature  
　　Description of term (English) → Measured temperature data (Celsius)
- データ前処理（日本語） → 異常値は除外する。
- Data preprocessing (English) → Outliers are excluded.
- 経時的に測定したデータ → Yes
- 行数 → 101
- 列数 → 5
- 同種のファイル概数 → 50
- 区切り文字 →  comma
- 文字コード →  ascii
- 備考（日本語） → データは毎日午前9時に収集
- Remarks (English) → Data is collected daily at 9 AM.

In [ ]:
async def _step(page):
    # テキストファイルアイテム追加ボタンをクリック
    await page.locator('h3:has-text("テキストファイル") ~ div button:has-text("テキストファイルアイテム追加")').first.click()
    time.sleep(1)
    
    # ファイル名、または命名規則・拡張子
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-file-name-convention-file-extension"]').fill('命名規則: [測定対象]_[日付].xlsx')
    
    # 説明（日本語）
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-description-jp"]').fill('心拍数の測定データ')
    # Description (English)
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-description-en"]').fill('heart rate measurement data.')
    
    # 行の説明 - アイテム追加
    await page.locator('h4:has-text("行の説明") ~ div button:has-text("アイテム追加")').first.click()
    time.sleep(0.5)
    # 行の位置（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-row|Position-of-row"]').fill('1行目')
    # 項目名（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-row|Name-of-term"]').fill('温度')
    # 項目の説明（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-row|Description-of-term"]').fill('測定された温度データ（摂氏）')
    # Position of row (English)
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-row|Position-of-row-en"]').fill('Row 1')
    # Name of term (English)
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-row|Name-of-term-en"]').fill('Temperature')
    
    # 列の説明 - アイテム追加
    await page.locator('h4:has-text("列の説明") ~ div button:has-text("アイテム追加")').first.click()
    time.sleep(0.5)
    # 列の位置（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-column|Position-of-column"]').fill('1列目')
    # 項目名（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-column|Name-of-term"]').fill('温度')
    # 項目の説明（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-column|Description-of-term"]').fill('測定された温度データ（摂氏）')
    # Position of column (English)
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-column|Position-of-column-en"]').fill('Column 1')
    # Name of term (English)
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-column|Name-of-term-en"]').fill('Temperature')
    # Description of term (English)
    await page.locator('textarea[name="__responseKey_d-txt-group|d-txt-description-of-column|Description-of-term-en"]').fill('Measured temperature data (Celsius)')
    
    # データ前処理（日本語）
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-data-preprocessing-jp"]').fill('異常値は除外する。')
    # Data preprocessing (English)
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-data-preprocessing-en"]').fill('Outliers are excluded.')
    
    # 経時的に測定したデータ - select dropdown
    await select_by_label(page, '経時的に測定したデータ', 'Yes')
    # 行数
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-number-of-rows"]').fill('101')
    # 列数
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-number-of-columns"]').fill('5')
    # 同種のファイル概数
    await page.locator('input[name="__responseKey_d-txt-group|d-txt-approximate-number-of-similar-files"]').fill('50')
    # 区切り文字
    await page.locator('input[name="__responseKey_d-txt-group|t-txt-delimiter"]').fill('comma')
    # 文字コード
    await page.locator('input[name="__responseKey_d-txt-group|t-txt-character-code"]').fill('ascii')
    
    # 備考（日本語）
    await page.locator('textarea[name="__responseKey_d-txt-group|t-txt-remarks-jp"]').fill('データは毎日午前9時に収集')
    # Remarks (English)
    await page.locator('textarea[name="__responseKey_d-txt-group|t-txt-remarks-en"]').fill('Data is collected daily at 9 AM.')

await run_pw(_step)

## エクセルファイルのエクセルファイルアイテム追加ボタンをクリックして、各項目を入力する（→以降を入力値とする）

- ファイル名、または命名規則・拡張子（e.g. xlsx, xlsm）など →  命名規則: [測定対象]_[日付].csv
- 説明（日本語） → 心拍数の測定データ
- Description (English) → heart rate measurement data.
- 行の説明 →   
　　行の位置（日本語）→　1行目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of row (English) → Row 1  
　　Name of term (English) → Temperature  
    Description of term (English) → Measured temperature data (Celsius)   
- 列の説明 →   
　　列の位置（日本語） → 1列目  
　　項目名（日本語） → 温度  
　　項目の説明（日本語） → 測定された温度データ（摂氏）  
　　Position of column (English) → Column 1  
　　Name of term (English) → Temperature  
　　Description of term (English) → Measured temperature data (Celsius)  
- データ前処理（日本語） → 異常値は除外する。
- Data Preprocessing (English) → Outliers are excluded.
- 経時的に測定したデータ → Yes
- 行数 → 51
- 列数 → 5
- 同種のファイル概数 → 10
- 備考（日本語） → データは毎日午前9時に収集
- Remarks (English) → Data is collected daily at 9 AM.

In [ ]:
async def _step(page):
    # エクセルファイルアイテム追加ボタンをクリック
    await page.locator('h3:has-text("エクセルファイル") ~ div button:has-text("エクセルファイルアイテム追加")').first.click()
    time.sleep(1)
    
    # ファイル名、または命名規則・拡張子
    await page.locator('input[name="__responseKey_d-exl-group|d-exl-file-name-convention-file-extension"]').fill('命名規則: [測定対象]_[日付].csv')
    
    # 説明（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-jp"]').fill('心拍数の測定データ')
    # Description (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-en"]').fill('heart rate measurement data.')
    
    # # 行の説明 - アイテム追加
    await page.locator('h4:has-text("行の説明") + div button:has-text("アイテム追加")').nth(1).click()
    time.sleep(0.5)
    # 行の位置（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-row|Position-of-row"]').fill('1行目')
    # 項目名（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-row|Name-of-term"]').fill('温度')
    # 項目の説明（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-row|Description-of-term"]').fill('測定された温度データ（摂氏）')
    # Position of row (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-row|Position-of-row-en"]').fill('Row 1')
    # Name of term (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-row|Name-of-term-en"]').fill('Temperature')
    # Description of term (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-row|Description-of-term-en"]').fill('Measured temperature data (Celsius)')
    
    # 列の説明 - アイテム追加
    await page.locator('h4:has-text("列の説明") + div button:has-text("アイテム追加")').nth(1).click()
    time.sleep(0.5)
    # 列の位置（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-column|Position-of-column"]').fill('1列目')
    # 項目名（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-column|Name-of-term"]').fill('温度')
    # 項目の説明（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-column|Description-of-term"]').fill('測定された温度データ（摂氏）')
    # Position of column (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-column|Position-of-column-en"]').fill('Column 1')
    # Name of term (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-column|Name-of-term-en"]').fill('Temperature')
    # Description of term (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|d-exl-description-of-column|Description-of-term-en"]').fill('Measured temperature data (Celsius)')
    
    # データ前処理（日本語）
    await page.locator('input[name="__responseKey_d-exl-group|d-exl-data-preprocessing-jp"]').fill('異常値は除外する。')
    # Data Preprocessing (English)
    await page.locator('input[name="__responseKey_d-exl-group|d-exl-data-preprocessing-en"]').fill('Outliers are excluded.')
    
    # 経時的に測定したデータ - select dropdown
    await select_by_label(page, '経時的に測定したデータ', 'Yes')
    # 行数
    await page.locator('input[name="__responseKey_d-exl-group|d-exl-number-of-rows"]').fill('51')
    # 列数
    await page.locator('input[name="__responseKey_d-exl-group|d-exl-number-of-columns"]').fill('5')
    # 同種のファイル概数
    await page.locator('input[name="__responseKey_d-exl-group|d-exl-approximate-number-of-similar-files"]').fill('10')
    
    # 備考（日本語）
    await page.locator('textarea[name="__responseKey_d-exl-group|t-exl-remarks-jp"]').fill('データは毎日午前9時に収集')
    # Remarks (English)
    await page.locator('textarea[name="__responseKey_d-exl-group|t-exl-remarks-en"]').fill('Data is collected daily at 9 AM.')

await run_pw(_step)

## 画像ファイルの画像ファイルアイテム追加ボタンをクリックして、各項目を入力する（→以降を入力値とする）

- ファイル名、または命名規則・拡張子（e.g. jpg, png, tif）など → 命名規則: [測定対象]_[日付].jpeg	
- 説明（日本語） → 実験Bの顕微鏡画像データ
- Description (English) → Microscope image data from Experiment B.
- データ前処理（日本語） → ノイズ除去、コントラスト調整
- Data preprocessing (English) → Noise removal, contrast adjustment
- 経時的に測定したデータ → Yes
- 幅ピクセル数 → 900 x 600
- 高さピクセル数 → 900 x 600 
- 解像度（水平方向） → 72dpi
- 解像度（垂直方向） → 72dpi 
- 色情報の数 → color
- 色ビット数（色深度） → 24ビット
- 圧縮形式 →非圧縮
- 同種のファイル概数 → 50
- 画像タイプ → jpeg
- 備考（日本語） → データは毎日午前9時に収集
- Remarks (English) →Data is collected daily at 9 AM.

In [ ]:
async def _step(page):
    # 画像ファイルアイテム追加ボタンをクリック
    await page.locator('h3:has-text("画像ファイル") ~ div button:has-text("画像ファイルアイテム追加")').first.click()
    time.sleep(1)
    
    # ファイル名、または命名規則・拡張子
    await page.locator('input[name="__responseKey_d-image-group|d-img-file-name-convention-file-extension"]').fill('命名規則: [測定対象]_[日付].jpeg')
    
    # 説明（日本語）
    await page.locator('input[name="__responseKey_d-image-group|d-img-description-jp"]').fill('実験Bの顕微鏡画像データ')
    # Description (English)
    await page.locator('input[name="__responseKey_d-image-group|d-img-description-en"]').fill('Microscope image data from Experiment B.')
    
    # データ前処理（日本語）
    await page.locator('input[name="__responseKey_d-image-group|d-img-data-preprocessing-jp"]').fill('ノイズ除去、コントラスト調整')
    # Data preprocessing (English)
    await page.locator('input[name="__responseKey_d-image-group|d-img-data-preprocessing-en"]').fill('Noise removal, contrast adjustment')
    
    # 経時的に測定したデータ - select dropdown
    await select_by_label(page, '経時的に測定したデータ', 'Yes')
    # 幅ピクセル数
    await page.locator('input[name="__responseKey_d-image-group|d-img-pixel-width"]').fill('900 x 600')
    # 高さピクセル数
    await page.locator('input[name="__responseKey_d-image-group|d-img-pixel-height"]').fill('900 x 600')
    # 解像度（水平方向）
    await page.locator('input[name="__responseKey_d-image-group|d-img-resolution-horizontal"]').fill('72 dpi')
    # 解像度（垂直方向）
    await page.locator('input[name="__responseKey_d-image-group|d-img-resolution-vertical"]').fill('72 dpi')
    # 色情報の数
    await page.locator('input[name="__responseKey_d-image-group|t-img-Color-Monochrome"]').fill('color')
    # 色ビット数（色深度）
    await page.locator('input[name="__responseKey_d-image-group|t-img-number-of-color-bit"]').fill('24ビット')
    # 圧縮形式
    await page.locator('input[name="__responseKey_d-image-group|t-img-compression-format"]').fill('非圧縮')
    # 同種のファイル概数
    await page.locator('input[name="__responseKey_d-image-group|d-img-approximate-number-of-similar-files"]').fill('50')
    # 画像タイプ
    await page.locator('input[name="__responseKey_d-image-group|t-img-image-type"]').fill('jpeg')

    # 備考（日本語）
    await page.locator('textarea[name="__responseKey_d-image-group|t-img-remarks-jp"]').fill('データは毎日午前9時に収集')
    # Remarks (English)
    await page.locator('textarea[name="__responseKey_d-image-group|t-img-remarks-en"]').fill('Data is collected daily at 9 AM.')

await run_pw(_step)

## 任意ファイルの各項目を入力する（→以降を入力値とする）

- ファイル名、または命名規則・拡張子など → 命名規則: [測定対象]_[日付].jpeg
- 説明（日本語） →  実験Bの顕微鏡画像データ
- Description (English) → Microscope image data from Experiment B.
- データ前処理（日本語） → ノイズ除去、コントラスト調整
- Data Preprocessing (English) → Noise removal, contrast adjustment
- 経時的に測定したデータ → Yes
- 行数 → 51
- 列数 → 5
- 同種のファイル概数 → 50
- テキスト/バイナリ → バイナリ
- ユーザー定義メタデータ項目 →   
　　メタデータ項目名 → サンプル保存条件  
　　値または内容 → Sample Storage Conditions  
　　Metadata item name → -80°C  
　　value or content → -80°C  
- 備考（日本語） →  データは毎日午前9時に収集
- Remarks (English) → Data is collected daily at 9 AM.

In [ ]:
async def _step(page):
    # 任意のファイルアイテム追加ボタンをクリック
    await page.locator('h3:has-text("任意のファイル") ~ div button:has-text("任意のファイルアイテム追加")').first.click()
    time.sleep(1)
    
    # ファイル名、または命名規則・拡張子など
    await page.locator('input[name="__responseKey_d-any-group|d-abt-file-name-convention-file-extension"]').fill('命名規則: [測定対象]_[日付].jpeg')
    
    # 説明（日本語）
    await page.locator('input[name="__responseKey_d-any-group|d-abt-description-jp"]').fill('実験Bの顕微鏡画像データ')
    # Description (English)
    await page.locator('input[name="__responseKey_d-any-group|d-abt-description-en"]').fill('Microscope image data from Experiment B.')
    
    # データ前処理（日本語）
    await page.locator('input[name="__responseKey_d-any-group|d-abt-data-preprocessing-jp"]').fill('ノイズ除去、コントラスト調整')
    # Data Preprocessing (English)
    await page.locator('input[name="__responseKey_d-any-group|d-abt-data-preprocessing-en"]').fill('Noise removal, contrast adjustment')
    
    # 経時的に測定したデータ - select dropdown
    await select_by_label(page, '経時的に測定したデータ', 'Yes')
    # 行数
    await page.locator('input[name="__responseKey_d-any-group|d-abt-number-of-rows"]').fill('51')
    # 列数
    await page.locator('input[name="__responseKey_d-any-group|d-abt-number-of-columns"]').fill('5')
    # 同種のファイル概数
    await page.locator('input[name="__responseKey_d-any-group|d-abt-approximate-number-of-similar-files"]').fill('50')
    # テキスト/バイナリ - select dropdown
    await select_by_label(page, 'テキスト/バイナリ', 'Binary | Binary')
    
    # ユーザー定義メタデータ項目 - アイテム追加
    await page.locator('h4:has-text("ユーザー定義メタデータ項目") ~ div button:has-text("アイテム追加")').first.click()
    time.sleep(0.5)
    # メタデータ項目名（日本語）
    await page.locator('textarea[name="__responseKey_d-any-group|t-abt-user-defined-metadata-items|Metadata-item-name"]').fill('サンプル保存条件')
    # 値または内容（日本語）
    await page.locator('textarea[name="__responseKey_d-any-group|t-abt-user-defined-metadata-items|value-or-content"]').fill('Sample Storage Conditions')
    # Metadata item name (English)
    await page.locator('textarea[name="__responseKey_d-any-group|t-abt-user-defined-metadata-items|Metadata-item-name-en"]').fill('-80°C')
    # value or content (English)
    await page.locator('textarea[name="__responseKey_d-any-group|t-abt-user-defined-metadata-items|value-or-content-en"]').fill('-80°C')
    
    # 備考（日本語）
    await page.locator('textarea[name="__responseKey_d-any-group|t-abt-remarks-jp"]').fill('データは毎日午前9時に収集')
    # Remarks (English)
    await page.locator('textarea[name="__responseKey_d-any-group|t-abt-remarks-en"]').fill('Data is collected daily at 9 AM.')

await run_pw(_step)

## データセットメタデータ登録画面で、次へボタンをクリックする

拡張メタデータアップロード画面が表示されること（ファイルリストが表示されるまで時間がかかります）

In [ ]:
async def _step(page):
    await page.locator('a[data-analytics-name="Go to next page"]').evaluate("el => el.click()")
    await expect(page.locator('h2:has-text("拡張メタデータアップロード")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 拡張メタデータアップロード画面で、＋ボタンをクリックし、Upload fileをクリック、ダイアログでtext2.csvを選択する

text2.csvがファイルリストにアップロードされること（ファイルリストが表示されるまで時間がかかります）

In [ ]:
async def _step(page):
    await page.locator('button[aria-label="Expand files menu"]').click()

    csv_file_name = 'text2.csv'
    csv_file_path = os.path.join('resources/Datatest-Metadataアドオン-未病データベース', csv_file_name)
    await page.locator('button:has-text("Upload file")').click()
    await page.wait_for_selector('input[type="file"]', state='attached')
    await page.set_input_files('//input[@type = "file"]', csv_file_path)

    await expect(page.locator('[data-test-file-name]:has-text("text2.csv")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 拡張メタデータアップロード画面で、次へボタンをクリックする

拡張メタデータアップの選択画面が表示されること（ファイルリストが表示されるまで時間がかかります）

In [ ]:
async def _step(page):
    await page.locator('a[data-analytics-name="Go to next page"]').evaluate("el => el.click()")
    await expect(page.locator('h2:has-text("拡張メタデータの選択")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 拡張メタデータ選択画面で、ファイルリストからimage1.jpegをチェックする。戻るボタンで、拡張メタデータアップロード画面に戻る。次へボタンで、再度 拡張メタデータ選択画面を表示する

image1.jpegがチェックされていること

In [ ]:
async def _step(page):
    await page.locator('//tr[.//p[normalize-space()="image1.jpeg"]]''//button[@data-test-file-metadata-input-upper]').click()
    await page.locator('a[data-analytics-name="Go to previous page"]').evaluate("el => el.click()")
    await expect(page.locator('h2:has-text("拡張メタデータアップロード")')).to_be_visible(timeout=transition_timeout)

    await page.locator('a[data-analytics-name="Go to next page"]').evaluate("el => el.click()")
    await expect(page.locator('h2:has-text("拡張メタデータの選択")')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('tr:has-text("image1.jpeg") i.fa-check-square')).to_be_visible()

await run_pw(_step)

## 拡張メタデータ選択画面で、画面左側のナビゲータで「チェックリスト（MS２共有）カタログ登録前」をクリックする

チェックリスト（MS２共有）カタログ登録前画面が表示されること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="チェックリスト:（MS2共有）カタログ登録前").click()
    await expect(page.locator('h2:has-text("チェックリスト:（MS2共有）カタログ登録前")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## チェックリスト（MS２共有）カタログ登録前画面で、すべてのチェック項目をチェックする

- チェックリストバージョンに2025/03/19が設定されていること
- カタログ登録前チェック日付に本日日付がセットされること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="チェックリスト:（MS2共有）カタログ登録前").click()
    all_checklists = page.locator('input[type="checkbox"][name^="__responseKey_Checklist"]')
    count = await all_checklists.count()
    for i in range(count):
        checkbox = all_checklists.nth(i)
        if not await checkbox.is_checked():
            await checkbox.check()

    today = datetime.now().strftime("%Y/%m/%d")
    await expect(page.locator('input[name="__responseKey_disclaimer-check-date"]')).to_have_value(today)

    await page.locator('input[name="__responseKey_disclaimer-version"]').scroll_into_view_if_needed()
    await expect(page.locator('input[name="__responseKey_disclaimer-version"]')).to_have_value("2025/03/19")
    time.sleep(1)

await run_pw(_step)

## 画面最上部の {プロジェクト名} > プロジェクトメタデータの登録 > の　｛プロジェクト名｝をクリックする

プロジェクトダッシュボード画面に戻ること

In [ ]:
async def _step(page):
    await page.locator('a[data-analytics-name="Go to review"]').click()
    await page.locator('a[data-analytics-name="Go to project"]').click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

- プロジェクトメタデータ参照の画面が表示されること
- 入力したプロジェクトメタデータが表示されること(最終更新日で確認する)

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}'))
    await expect(card).to_be_visible(timeout=transition_timeout)

    today_str = datetime.now().strftime("%b %d %Y")
    updated = card.locator('p[data-test-time-updated]')
    await expect(updated).to_be_visible(timeout=transition_timeout)  
    await expect(updated).to_contain_text(today_str)

await run_pw(_step)

## 下書きタブのリストから、№87で作成した項目の「エクスポート」ボタンをクリックする

エクスポート形式の選択ダイアログが表示されること

In [ ]:
async def _step(page):
    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}'))
    export_button = card.locator('button[data-analytics-name="Export"]')
    await expect(export_button).to_be_visible(timeout=transition_timeout)
    await export_button.click()

    modal = page.locator('[data-analytics-scope="RegistrationReport modal"]')
    await expect(modal).to_be_visible()
    await expect(modal.locator('h4', has_text='エクスポート形式の選択')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## ブラウザの別タブでアドレスに https://ams.ir.example を入力してJAIRO Cloudのページをオープンすること

JAIRO Cloudとのセッション状況によって、ログイン画面となった場合は下記アカウントでログインを実施すること  
- メールアドレス：(個別に情報共有)
- パスワード：(個別に情報共有)

In [ ]:
async def _step(page):
    new_page = await page.context.new_page()
    await new_page.goto(weko_url)
    login_btn = new_page.get_by_role('button', name='ログイン')
    if await login_btn.is_visible():
        await login_btn.click()
        await new_page.locator('#email').fill(weko_admin_email)
        await new_page.locator('#password').fill(weko_admin_password)
        await new_page.get_by_role("button", name="ログイン").click()
    
        await expect(new_page.locator("button", has_text="Authorize application")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## GakuNinRDMのタブに戻り、エクスポート形式の選択ダイアログのプルダウンから｛リポジトリ（JAIRO Cloud）に登録 - New Index ｝を選択して、エクスポートボタンをクリックする（1分程度かかります）	

- 「エクスポート結果」ダイアログにて、「登録が成功しました。」メッセージおよびリンクが表示されること

In [ ]:
metadata_url = None
export_result_link = None

async def _step(page):
    await page.bring_to_front()
    time.sleep(2)
    
    # Select export format and click submit
    await page.locator('#registration-report-format-selection').select_option(label=f'リポジトリ (JAIRO Cloud)に登録 - {weko_index_name}')
    await page.locator('button[data-test-registration-report-submit]').click()
    
    # Wait for result dialog to appear
    result_modal = page.locator('[data-analytics-scope="RegistrationReport result modal"]')
    await expect(result_modal).to_be_visible(timeout=transition_timeout*2)  # 2 minutes timeout for export
    
    # Verify success message
    success_message = result_modal.locator('div[class*="_Export__description_"]')
    await expect(success_message).to_contain_text('登録が成功しました。', timeout=transition_timeout)
    print("エクスポート結果ダイアログ: '登録が成功しました。'")
    
    # Get the link to JAIRO Cloud page
    result_link = result_modal.locator('a[data-test-registration-report-result-link]')
    await expect(result_link).to_be_visible(timeout=transition_timeout)
    print(f"JAIRO Cloudへのリンク: {await result_link.get_attribute('href')}")
    global metadata_url, export_result_link
    metadata_url = page.url
    export_result_link = await result_link.get_attribute('href')
    
    if idp_name_1 == 'FakeCAS':
        return
    
    # Setup wait for new page before clicking link
    popup_future = page.context.wait_for_event('page')
    await result_link.click()
    
    # Wait for JAIRO Cloud popup page
    popup = await popup_future
    await popup.wait_for_load_state()
    time.sleep(5)  # Wait for page to fully load
    
    # Wait for detail table to be visible
    await expect(popup.locator('table.detail-table')).to_be_visible(timeout=transition_timeout)
    print("JAIRO Cloudページがオープン - detail-table表示確認")
    
    # Define expected values based on 未病データベース_プロジェクトメタデータ sheet
    # Format: (section_header, expected_value)
    expected_items = [
        # Title of Dataset
        ("Title of Dataset", rdm_project_name),
        # MSProject name - Award Title
        ("MSProject name", "MS2合原PJ｜MS2 Aihara PJ"),
        # Data Id
        ("Data Id", "ms2dataid"),
        # Purpose of experiment
        ("Purpose of experiment", "細胞成長のメカニズムを解明するため"),
        ("Purpose of experiment", "To elucidate the mechanisms of cell growth"),
        # Description of experimental condition
        ("Description of experimental condition", "実験は2025年3月1日から3月31日まで行われ、顕微鏡と温度計を使用して細胞の成長を観察しました。"),
        ("Description of experimental condition", "The experiment was conducted from March 1 to March 31, 2025, using a microscope and thermometer to observe cell growth."),
        # Keywords
        ("Keywords", "細胞成長"),
        ("Keywords", "Cell growth"),
        # Dataset Research field
        ("Dataset Research field", "ライフサイエンス｜Life Science"),
        # Analysis type
        ("Analysis type", "配列データ｜Sequence data"),
        ("Analysis type", "エピゲノムデータ｜Epigenome data"),
        ("Analysis type", "トランスクリプトームデータ｜Transcriptome data"),
        ("Analysis type", "プロテオームデータ｜Proteome data"),
        ("Analysis type", "リン酸化プロテオームデータ｜Phosphoproteome data"),
        ("Analysis type", "メタボロームデータ｜Metabolome data"),
        ("Analysis type", "イメージデータ｜Imaging data"),
        ("Analysis type", "時系列データ｜Time series data"),
        ("Analysis type", "経時的データ｜Over time data"),
        ("Analysis type", "その他｜Other"),
        # Analysis type (Other)
        ("Analysis type (Other)", "フェノームデータ"),
        # The presence of metadata files
        ("The presence of metadata files", "有｜Yes"),
        # (Yes) Metadata file name(s)
        ("(Yes) Metadata file name(s)", "フェノームデータ_メタデータ_20250321.txt"),
        # Necessity of Contact and Permission
        ("Necessity of Contact and Permission", "連絡不要｜No contact required"),
        # Necessity of including in Acknowledgments
        ("Necessity of including in Acknowledgments", "不要｜Unnecessary"),
        # Names to be included in the Acknowledgments
        ("Names to be included in the Acknowledgments", "TEST"),
        ("Names to be included in the Acknowledgments", "Taro Tanaka、Hanako Yamada"),
        # Other Conditions or Special Notes
        ("Other Conditions or Special Notes", "本データは特定のソフトウェアバージョンでのみ解析可能です。"),
        ("Other Conditions or Special Notes", "This data can only be analyzed with a specific software version."),
        # License
        ("License", "ライセンスなし｜No license"),
        # Pay or Free
        ("Pay or Free", "無償｜Free"),
        # Availability of commercial use
        ("Availability of commercial use", "否｜No"),
        # Access Rights
        ("Access Rights", "restricted access"),
        # Scheduled release date
        ("Scheduled release date", weko_scheduled_release_date),
        # Repository information
        ("Repository information", "GakuNin RDM"),
        # Repository URL/ DOI link
        ("Repository URL/ DOI link", "https://doi.org/10.1234/example1"),
        # Other supplementary information
        ("Other supplementary information", "データは特定の条件下でのみ使用可能です。"),
        ("Other supplementary information", "Data can only be used under specific conditions."),
        # data creator - values in popover and td
        ("data creator", "Ichiro Yamamoto"),
        ("data creator", "山本一郎"),
        ("data creator", "東西大学 生物学部"),
        ("data creator", "Department of Biology, Tozai University"),
        ("data creator", "ichiro.yamamoto@example.com"),
        # data manager
        ("data manager", "鈴木花子"),
        ("data manager", "Hanako Suzuki"),
        ("data manager", "南北大学 化学部"),
        ("data manager", "Department of Chemistry, Nanboku University"),
        ("data manager", "hanako.suzuki@example.com"),
        # Target type of acquired data
        ("Target type of acquired data", "ヒト細胞"),
        ("Target type of acquired data", "Human cells"),
        # Ethics Review Committee Approval
        ("Ethics Review Committee Approval", "承認済み（2025年2月15日）"),
        ("Ethics Review Committee Approval", "Approved (February 15, 2025)"),
        # (Human) Informed Consent (IC)
        ("(Human) Informed Consent (IC)", "有｜Yes"),
        # (IC is Yes) Consent for provision to a third party
        ("(IC is Yes) Consent for provision to a third party", "無｜No"),
        # (IC is Yes) Overseas Offerings
        ("(IC is Yes) Overseas Offerings", "有｜Yes"),
        # (IC is Yes) Industrial use, etc.
        ("(IC is Yes) Industrial use, etc.", "無｜No"),
        # (IC is No)
        ("(IC is No)", "同意不要｜Agree not to"),
        # (Human) Anonymous processing
        ("(Human) Anonymous processing", "有｜Yes"),
        # conflict of interest Yes or No
        ("conflict of interest Yes or No", "有｜Yes"),
        # conflict of interest (Yes)
        ("conflict of interest (Yes)", "XYZ株式会社"),
        ("conflict of interest (Yes)", 'Company "XYZ Corporation"'),
        # Remarks
        ("Remarks", "利益相反の詳細は、プロジェクトの資金提供者が実験結果に影響を与える可能性があることです。"),
        ("Remarks", 'Details of the conflict of interest include the possibility that the project funder, the fictional company "ABC Corporation," may influence the experimental results.'),
    ]
    
    print("\n=== JAIRO Cloud プロジェクトメタデータ検証 ===")
    failed_checks = []
    passed_count = 0
    
    # Get the full page content for searching
    page_content = await popup.content()
    
    for section_header, expected_value in expected_items:
        try:
            found = False
            
            # Method 1: Check if value exists in page content directly
            if expected_value in page_content:
                found = True
            else:
                # Method 2: Find value in td.multiple-line cells
                value_cell = popup.locator(f'td.multiple-line:has-text("{expected_value}")')
                if await value_cell.count() > 0:
                    found = True
                else:
                    # Method 3: Check in anchor tags for URLs
                    link_cell = popup.locator(f'td a:has-text("{expected_value}")')
                    if await link_cell.count() > 0:
                        found = True
                    else:
                        # Method 4: Check in creator popover content
                        creator_cell = popup.locator(f'.popover-content:has-text("{expected_value}")')
                        if await creator_cell.count() > 0:
                            found = True
                        else:
                            # Method 5: Check in td.key_lang for creator info
                            key_lang_cell = popup.locator(f'td.key_lang:has-text("{expected_value}")')
                            if await key_lang_cell.count() > 0:
                                found = True
            
            if found:
                print(f"[{section_header}]: '{expected_value}' - Found")
                passed_count += 1
            else:
                print(f"[{section_header}]: '{expected_value}' - NOT Found")
                failed_checks.append(f"{section_header}: expected '{expected_value}'")
        except Exception as e:
            print(f"[{section_header}]: Error checking '{expected_value}' - {str(e)}")
            failed_checks.append(f"{section_header}: Error - {str(e)}")
    
    # Summary
    print(f"\n=== 検証結果 ===")
    print(f"Total checked: {len(expected_items)}")
    print(f"Passed: {passed_count}")
    print(f"Failed: {len(failed_checks)}")
    
    if failed_checks:
        print("\n失敗した項目:")
        for item in failed_checks:
            print(f"  - {item}")
    
    assert len(failed_checks) == 0, f"Some metadata values were not found on JAIRO Cloud page: {failed_checks}"
    
    print("\n=== すべてのプロジェクトメタデータがJAIRO Cloudに正しくセットされました ===")

await run_pw(_step)

## ExportのURLを開く

- JAIRO Cloudのページがオープンすること
- プロジェクトメタデータの項目がJAIRO Cloudの項目としてセットされていること（未病データベース_プロジェクトメタデータシートの項目で確認すること）

In [ ]:
SCROLL_ITEMS = [
    "MSProject name",
    "Award Title",
    "Data Id",
    "Related Identifier",
    "Date (Registered) in Metadata",
    "Date (Updated) in Metadata",
    "Purpose of experiment (Purpose of Analytical results or Analytical tools)",
    "Description of experimental condition (Description of Analytical results or Analytical tools)",
    "Keywords",
    "Dataset Research field",
    "Analysis type",
    "Analysis type (Other)",
    "The presence of metadata files created for a specific modality in other databases",
    "(Yes) Metadata file name(s)",
    "Necessity of Contact and Permission",
    "Necessity of including in Acknowledgments",
    "Names to be included in the Acknowledgments (individuals, groups, etc.), or the full text of the Acknowledgments",
    "Other Conditions or Special Notes",
    "Scheduled release date (If access rights are not public)",
    "Repository information",
    "Repository URL/ DOI link",
    "Other supplementary information",
    "data creator",
    "Contributor Name",
    "Affiliation",
    "Contributor Email Address",
    "Target type of acquired data",
    "Ethics Review Committee Approval",
    "(Human) Informed Consent (IC) Yes, No, or Unnecessary",
    "(IC is Yes) Consent for provision to a third party",
    "(IC is Yes) Overseas Offerings",
    "(IC is Yes) Industrial use, etc.",
    "(IC is No)",
    "(Human) Anonymous processing Yes or No",
    "conflict of interest Yes or No",
    "conflict of interest (Yes)",
    "Remarks",
    "ProjectURL",
    "Extra",
    "Publish Status"
]

In [ ]:
async def _step(page):
    await page.goto(export_result_link)
    await asyncio.sleep(5)
    await page.get_by_role("button", name="That's ok").click()
    for text in SCROLL_ITEMS:
        locator = page.locator(f"text={text}").first
        if await locator.count() > 0:
            await locator.scroll_into_view_if_needed()
            await asyncio.sleep(0.2)
    return page

await run_pw(_step)

## GakuNinRDMのタブに戻り、プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

入力したプロジェクトメタデータが表示されること(最終更新日で確認する)

In [ ]:
async def _step(page):
    await page.goto(metadata_url)
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}'))
    await expect(card).to_be_visible(timeout=transition_timeout)

    today_str = datetime.now().strftime("%b %d %Y")
    updated = card.locator('p[data-test-time-updated]')
    await expect(updated).to_be_visible(timeout=transition_timeout)  
    await expect(updated).to_contain_text(today_str)

await run_pw(_step)

## 入力したプロジェクトメタデータの「編集ボタン」をクリックする

プロジェクトメタデータ編集画面が入力済の状態でオープンされること

In [ ]:
async def _step(page):
    await page.locator('a[data-analytics-name="Edit"]').click()
    await expect(page.locator('h2:has-text("プロジェクトメタデータ登録")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「クリップボードにコピー」ボタンをクリックする

画面右上に「クリップボードコピーされました」のトーストが出ること（プログレスバーがいっぱいになって、トーストが消えること）
※No.98で貼り付けするまでクリップボードを更新しないように注意"

In [ ]:
import json
clipboard_content = None

async def _step(page):
    global clipboard_content
    await mock_clipboard(page)
    await page.locator('button[data-analytics-name="Copy to clipboard"]').click()
    clipboard_content = await get_mocked_clipboard_text(page)
    clipboard_json = json.loads(clipboard_content)
    assert clipboard_json["project-name"] == "MS2合原PJ|MS2 Aihara PJ"

await run_pw(_step)

## 画面最上部の {プロジェクト名} > プロジェクトメタデータの登録 > の　｛プロジェクト名｝をクリックする

プロジェクトダッシュボード画面に戻ること

In [ ]:
async def _step(page):
    await page.locator('a[data-analytics-name="Go to project"]').click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

プロジェクトメタデータ参照の画面（メタデータタブ、下書きタブで構成）が表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}'))
    await expect(card).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトメタデータ参照の画面の「新規メタデータを作成」ボタンをクリックする

メタデータ様式の選択を求めるダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-metadata-button]').click()

    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## メタデータ様式の選択を求めるダイアログで「ムーンショット目標2未病データベース（未病DB）のメタデータ登録 」のラジオボタンを選択して「メタデータを作成」をクリックする

メタデータの各項目を入力する画面が表示されること

In [ ]:
async def _step(page):
    await page.get_by_text( "ムーンショット目標2データベース（未病DB）のメタデータ登録").first.click()
    await page.locator('//*[@data-test-new-report-modal-create-report-button]').click()

    await expect( page.locator('[data-test-page-heading]', has_text='プロジェクトメタデータ登録')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面右側の「クリップボードから貼り付け」をクリックする、そのあとに「貼り付けボタン」が有効になったらクリックする

- 2～3秒してsubmit（画面更新）されること
- プロジェクトメタデータ登録ページに「未病データベース_プロジェクトメタデータ」シートの値がセットされていること

In [ ]:
async def _step(page):
    await mock_clipboard(page)
    # Set clipboard content page's mocked clipboard before pasting
    await page.evaluate("(text) => { navigator.clipboard._text = text; }", clipboard_content)
    await page.locator('button[data-analytics-name="Paste from clipboard"]').click()
    time.sleep(3)

    # Verify values from "未病データベース_プロジェクトメタデータ" sheet are set
    print("\n=== プロジェクトメタデータ登録ページの値検証 ===")
    failed_checks = []
    passed_count = 0
    
    # ===== TEXT INPUT FIELDS =====
    text_input_fields = [
        # データセットの名称（日本語）
        ('__responseKey_title-of-dataset', rdm_project_name, 'データセットの名称（日本語）'),
        # Title of Dataset (English)
        ('__responseKey_title-of-dataset-en', rdm_project_name, 'Title of Dataset (English)'),
        # データID
        ('__responseKey_data-id', 'ms2dataid', 'データID'),
        # 実験（もしくは解析結果・解析ツールなど）の目的（日本語）
        ('__responseKey_purpose-of-experiment', '細胞成長のメカニズムを解明するため', '実験の目的（日本語）'),
        # Purpose of experiment (English)
        ('__responseKey_purpose-of-experiment-en', 'To elucidate the mechanisms of cell growth', 'Purpose of experiment (English)'),
        # 実験状況などの説明（日本語）
        ('__responseKey_description-of-experimental-condition', '実験は2025年3月1日から3月31日まで行われ、顕微鏡と温度計を使用して細胞の成長を観察しました。', '実験状況の説明（日本語）'),
        # Description of experimental condition (English)
        ('__responseKey_description-of-experimental-condition-en', 'The experiment was conducted from March 1 to March 31, 2025, using a microscope and thermometer to observe cell growth.', 'Description of experimental condition (English)'),
        # キーワード（日本語）
        ('__responseKey_keywords|filename', '細胞成長', 'キーワード（日本語）'),
        # Keywords (English)
        ('__responseKey_keywords|filename-en', 'Cell growth', 'Keywords (English)'),
        # メタデータファイル名
        ('__responseKey_metadata-filename|filename', 'フェノームデータ_メタデータ_20250321.txt', 'メタデータファイル名'),
        # 謝辞に記載する名称（日本語）
        ('__responseKey_names-to-be-included-in-the-acknowledgments', 'TEST', '謝辞に記載する名称（日本語）'),
        # Names to be included in the Acknowledgments (English)
        ('__responseKey_names-to-be-included-in-the-acknowledgments-en', 'Taro Tanaka、Hanako Yamada', 'Names to be included in the Acknowledgments (English)'),
        # その他条件（日本語）
        ('__responseKey_other-conditions-or-special-notes', '本データは特定のソフトウェアバージョンでのみ解析可能です。', 'その他条件（日本語）'),
        # Other Conditions or Special Notes (English)
        ('__responseKey_other-conditions-or-special-notes-en', 'This data can only be analyzed with a specific software version.', 'Other Conditions or Special Notes (English)'),
        # 公開予定日
        ('__responseKey_scheduled-release-date', scheduled_release_date, '公開予定日'),
        # リポジトリURL・DOIリンク
        ('__responseKey_repository-url-doi-link', 'https://doi.org/10.1234/example1', 'リポジトリURL・DOIリンク'),
        # データ作成者 氏名（日本語）
        ('__responseKey_data-creator|name', '山本一郎', 'データ作成者 氏名（日本語）'),
        # Name of data creator (English)
        ('__responseKey_data-creator|name-en', 'Ichiro Yamamoto', 'Name of data creator (English)'),
        # データ作成者 所属（日本語）
        ('__responseKey_data-creator|belonging', '東西大学 生物学部', 'データ作成者 所属（日本語）'),
        # Belonging of data creator (English)
        ('__responseKey_data-creator|belonging-en', 'Department of Biology, Tozai University', 'Belonging of data creator (English)'),
        # データ作成者 連絡先
        ('__responseKey_data-creator|contact', 'ichiro.yamamoto@example.com', 'データ作成者 連絡先'),
        # データ管理者 氏名（日本語）
        ('__responseKey_data-manager|name', '鈴木花子', 'データ管理者 氏名（日本語）'),
        # Name of data manager (English)
        ('__responseKey_data-manager|name-en', 'Hanako Suzuki', 'Name of data manager (English)'),
        # データ管理者 所属（日本語）
        ('__responseKey_data-manager|belonging', '南北大学 化学部', 'データ管理者 所属（日本語）'),
        # Belonging of data manager (English)
        ('__responseKey_data-manager|belonging-en', 'Department of Chemistry, Nanboku University', 'Belonging of data manager (English)'),
        # データ管理者 連絡先
        ('__responseKey_data-manager|contact', 'hanako.suzuki@example.com', 'データ管理者 連絡先'),
        # 取得データの対象種別（日本語）
        ('__responseKey_target-type-of-acquired-data', 'ヒト細胞', '取得データの対象種別（日本語）'),
        # Target type of acquired data (English)
        ('__responseKey_target-type-of-acquired-data-en', 'Human cells', 'Target type of acquired data (English)'),
        # 倫理審査委員会承認（日本語）
        ('__responseKey_ethics-review-committee-approval', '承認済み（2025年2月15日）', '倫理審査委員会承認（日本語）'),
        # Ethics Review Committee Approval (English)
        ('__responseKey_ethics-review-committee-approval-en', 'Approved (February 15, 2025)', 'Ethics Review Committee Approval (English)'),
        # 利益相反の有無（有の場合）（日本語）
        ('__responseKey_conflict-of-interest', 'XYZ株式会社', '利益相反の有無（有の場合）（日本語）'),
        # conflict of interest (Yes) (English)
        ('__responseKey_conflict-of-interest-en', 'Company "XYZ Corporation"', 'conflict of interest (Yes) (English)'),
    ]
    
    # ===== TEXTAREA FIELDS =====
    textarea_fields = [
        # 解析対象データ（その他）
        ('__responseKey_Analysis-type-other', 'フェノームデータ', '解析対象データ（その他）'),
        # その他補足事項（日本語）
        ('__responseKey_other-supplementary-information', 'データは特定の条件下でのみ使用可能です。', 'その他補足事項（日本語）'),
        # Other supplementary information (English)
        ('__responseKey_other-supplementary-information-en', 'Data can only be used under specific conditions.', 'Other supplementary information (English)'),
        # 備考（日本語）
        ('__responseKey_remarks-3', '利益相反の詳細は、プロジェクトの資金提供者が実験結果に影響を与える可能性があることです。', '備考（日本語）'),
        # Remarks (English)
        ('__responseKey_remarks-3-en', 'Details of the conflict of interest include the possibility that the project funder, the fictional company "ABC Corporation," may influence the experimental results.', 'Remarks (English)'),
    ]
    
    # ===== DROPDOWN FIELDS =====
    dropdown_fields = [
        # MSプロジェクト名
        ('MSプロジェクト名', 'MS2合原PJ|MS2 Aihara PJ'),
        # データセットの分野
        ('データセットの分野', 'ライフサイエンス|Life Science'),
        # 他のデータベースで特定のモダリティ用に作成したメタデータファイルの有無
        ('メタデータファイルの有無', '有|Yes'),
        # 連絡・許諾の要不要
        ('連絡・許諾の要不要', '連絡不要|No contact required'),
        # 謝辞に記載の要不要
        ('謝辞に記載の要不要', '不要|Unnecessary'),
        # ライセンス
        ('ライセンス', 'ライセンスなし|No license'),
        # 有償・無償
        ('有償・無償', '無償|Free'),
        # 商用利用の可否
        ('商用利用の可否', '否|No'),
        # アクセス権
        ('アクセス権', '共有|restricted access'),
        # リポジトリ情報
        ('リポジトリ情報', 'GakuNin RDM'),
        # インフォームドコンセント
        ('インフォームドコンセント', '有|Yes'),
        # 第三者提供の同意
        ('第三者提供の同意', '無|No'),
        # 海外提供
        ('海外提供', '有|Yes'),
        # 産業利用等
        ('産業利用等', '無|No'),
        # IC無の場合
        ('IC無の場合', '同意不要|Agree not to'),
        # 匿名加工の有無
        ('匿名加工の有無', '有|Yes'),
        # 利益相反の有無
        ('利益相反の有無', '有|Yes'),
    ]
    
    # ===== CHECKBOX FIELDS (解析対象データ) =====
    analysis_type_checkboxes = [
        '配列データ|Sequence data',
        'エピゲノムデータ|Epigenome data',
        'トランスクリプトームデータ|Transcriptome data',
        'プロテオームデータ|Proteome data',
        'リン酸化プロテオームデータ|Phosphoproteome data',
        'メタボロームデータ|Metabolome data',
        'イメージデータ|Imaging data',
        '時系列データ|Time series data',
        '経時的データ|Over time data',
        'その他|Other',
    ]
    
    # Verify text input fields
    print("\n--- TEXT INPUT FIELDS ---")
    for field_name, expected_value, label in text_input_fields:
        try:
            input_elem = page.locator(f'input[name="{field_name}"]')
            actual_value = await input_elem.input_value()
            if actual_value == expected_value:
                print(f"[{label}]: '{expected_value}' - OK")
                passed_count += 1
            else:
                print(f"[{label}]: Expected '{expected_value}', Got '{actual_value}'")
                failed_checks.append(f"{label}: expected '{expected_value}', got '{actual_value}'")
        except Exception as e:
            print(f"[{label}]: Error - {str(e)}")
            failed_checks.append(f"{label}: Error - {str(e)}")
    
    # Verify textarea fields
    print("\n--- TEXTAREA FIELDS ---")
    for field_name, expected_value, label in textarea_fields:
        try:
            textarea_elem = page.locator(f'textarea[name="{field_name}"]')
            actual_value = await textarea_elem.input_value()
            if actual_value == expected_value:
                print(f"[{label}]: OK")
                passed_count += 1
            else:
                print(f"[{label}]: Expected '{expected_value[:30]}...', Got '{actual_value[:30]}...'")
                failed_checks.append(f"{label}: value mismatch")
        except Exception as e:
            print(f"[{label}]: Error - {str(e)}")
            failed_checks.append(f"{label}: Error - {str(e)}")
    
    # Verify dropdown fields
    print("\n--- DROPDOWN FIELDS ---")
    for field_name, expected_value in dropdown_fields:
        try:
            dropdown = page.locator(f'.ember-power-select-selected-item:has-text("{expected_value}")')
            if await dropdown.count() > 0:
                print(f"[{field_name}]: '{expected_value}' - OK")
                passed_count += 1
            else:
                print(f"[{field_name}]: Expected '{expected_value}' - NOT Found")
                failed_checks.append(f"{field_name}: expected '{expected_value}'")
        except Exception as e:
            print(f"[{field_name}]: Error - {str(e)}")
            failed_checks.append(f"{field_name}: Error - {str(e)}")
    
    # Verify checkbox fields (解析対象データ)
    print("\n--- CHECKBOX FIELDS (解析対象データ) ---")
    for checkbox_label in analysis_type_checkboxes:
        try:
            checkbox = page.locator(f'div[data-test-multiple-select-option="{checkbox_label}"] input[type="checkbox"]')
            if await checkbox.count() > 0:
                is_checked = await checkbox.is_checked()
                if is_checked:
                    print(f"[解析対象データ - {checkbox_label}]: Checked - OK")
                    passed_count += 1
                else:
                    print(f"[解析対象データ - {checkbox_label}]: NOT Checked")
                    failed_checks.append(f"解析対象データ - {checkbox_label}: NOT Checked")
            else:
                print(f"? [解析対象データ - {checkbox_label}]: Checkbox not found")
        except Exception as e:
            print(f"[解析対象データ - {checkbox_label}]: Error - {str(e)}")
            failed_checks.append(f"解析対象データ - {checkbox_label}: Error - {str(e)}")
    
    # Summary
    total_checks = len(text_input_fields) + len(textarea_fields) + len(dropdown_fields) + len(analysis_type_checkboxes)
    print(f"\n=== 検証結果 ===")
    print(f"Total checked: {total_checks}")
    print(f"Passed: {passed_count}")
    print(f"Failed: {len(failed_checks)}")
    
    if failed_checks:
        print("\n失敗した項目:")
        for item in failed_checks:
            print(f"  - {item}")
    
    assert len(failed_checks) == 0, f"Some metadata values were not found on the page: {failed_checks}"
    
    print("\n=== プロジェクトメタデータがすべて正しくセットされました ===")

await run_pw(_step)

## 画面右上の「自動保存済：xxxxxx」の記述が ”a few seconds ago" から "a minute ago" に変わるまでまってから、データセットの名称（日本語）の末尾に”修正”と追記すること（追記後はフォーカスを移すこと）

「自動保存済：xxxxxx」の記述が ”a few seconds ago"に変わる事

In [ ]:
async def _step(page):
    await page.locator('input[name="__responseKey_title-of-dataset"]').fill(f'{rdm_project_name}-修正')
    await page.keyboard.press('Enter')
    time.sleep(2)
    await expect(page.locator('span:has-text("自動保存済み")')).to_contain_text('a few seconds ago', timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

ファイル画面が表示される

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await page.locator('#projectNavFiles a').click()
    time.sleep(1)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## ファイル一覧の「NII Storage」から「text1.csv」をクリックしてメニューから「メタデータ削除」をクリックする

「ファイルメタデータの削除」のダイアログが出力される

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, csv_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ削除"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ削除"]').click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ファイルメタデータの削除」のダイアログで「削除」をクリックする

ファイル名から{}が消える

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "btn-danger") and text() = "削除"]').first.click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の「NII Storage」から「text1.tsv」をクリックしてメニューから「メタデータ削除」をクリックする

「ファイルメタデータの削除」のダイアログが出力される

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, tsv_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ削除"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ削除"]').click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ファイルメタデータの削除」のダイアログで「削除」をクリックする

ファイル名から{}が消える

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "btn-danger") and text() = "削除"]').first.click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の「NII Storage」から「excel1.xlsx」をクリックしてメニューから「メタデータ削除」をクリックする

「ファイルメタデータの削除」のダイアログが出力される

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, excel_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ削除"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ削除"]').click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ファイルメタデータの削除」のダイアログで「削除」をクリックする

ファイル名から{}が消える

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "btn-danger") and text() = "削除"]').first.click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の「NII Storage」から「image1.jpeg」をクリックしてメニューから「メタデータ削除」をクリックする

「ファイルメタデータの削除」のダイアログが出力される

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, image_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ削除"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ削除"]').click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ファイルメタデータの削除」のダイアログで「削除」をクリックする

ファイル名から{}が消える

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "btn-danger") and text() = "削除"]').first.click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の「NII Storage」から「image1.png」をクリックしてメニューから「メタデータ削除」をクリックする

「ファイルメタデータの削除」のダイアログが出力される

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, png_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ削除"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ削除"]').click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ファイルメタデータの削除」のダイアログで「削除」をクリックする

ファイル名から{}が消える

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "btn-danger") and text() = "削除"]').first.click()
    await expect(page.locator('//*[text() = "メタデータを削除してよろしいですか？この操作は元に戻せません。"]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の「NII Storage」から「text1.csv」「text1.tsv」「excel1.xlsx」「image1.jpeg」「image1.png」「text2.csv」を複数選択する（Shiftキーを押しながら、ファイル名をクリックする）上部メニューから「複数削除」をクリックする

「複数のファイルを削除しますか？」のダイアログが出力される

In [ ]:
files = [
    "excel1.xlsx",
    "image1.jpeg",
    "image1.png",
    "text1.csv",
    "text1.tsv",
    "text2.csv",
]

async def _step(page):
    first = grdm.get_select_file_extension_locator(page, files[0])
    await first.scroll_into_view_if_needed()
    await first.click()

    await page.keyboard.down("Shift")
    
    for name in files[1:]:
        locator = grdm.get_select_file_extension_locator(page, name)
        await locator.scroll_into_view_if_needed()
        await locator.click()
    
    await page.keyboard.up("Shift")

    await expect(page.locator('//*[text() = "複数削除"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "複数削除"]').click()

    modal = page.locator('.modal-content')
    await expect(modal.get_by_role("heading", name="複数のファイルを削除しますか？")).to_be_visible(timeout=transition_timeout)
        
await run_pw(_step)

## 「複数のファイルを削除しますか？」のダイアログで「全て削除」をクリックする

ファイル一覧から選択したファイルが削除される

In [ ]:
async def _step(page):
    modal = page.locator('.modal-content')
    await modal.locator('span.btn.btn-danger:has-text("全て削除")').click()

    for name in files:
        await expect(page.locator('.fg-file-links', has_text=name)).to_have_count(0)

await run_pw(_step)

## 上部メニューから「メタデータ」をクリックする

- プロジェクトメタデータ参照の画面が表示されること
- プロジェクトメタデータのリストが表示される

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}'))
    await expect(card).to_have_count(2, timeout=transition_timeout)

await run_pw(_step)

## 作成した項目の「削除」ボタンをクリックする

「確認してください。」ダイアログが表示される

In [ ]:
async def _step(page):
    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}')).filter(has_text="登録済")
    await card.get_by_role("button", name="削除").click()

    modal = page.locator('.modal-dialog')
    await expect(modal.get_by_role("heading", name="確認してください。")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「確認してください。」ダイアログの「削除」ボタンをクリックする

下書きタブのリストから、№87で作成した項目が削除される

In [ ]:
async def _step(page):
    modal = page.locator(".modal-dialog")
    await modal.get_by_role("button", name="削除").click()

    card = page.locator('[data-test-draft-registration-card]',has=page.locator(f'text={rdm_project_name}')).filter(has_text="登録済")
    await expect(card).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのJAIRO Cloud(https://ams.ir.example/search)のタブを開く

JAIRO Cloudの画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url)
    if idp_name_1 == 'FakeCAS':
        await page.locator('//div[contains(@class, "navbar-form")]//button[contains(@class, "dropdown-toggle")]').click()
        logout_btn = page.locator('//a[@href = "/logout/"]')
        await expect(logout_btn).to_be_visible(timeout=transition_timeout)
    else:
        logout_btn = page.get_by_role('button', name='ログアウト')

    if await logout_btn.is_visible(timeout=transition_timeout):
        print('Logged in')
        return

    await page.get_by_role('button', name='ログイン').click()
    await page.locator('input[name="email"]').fill(weko_admin_email)
    await page.locator('input[name="password"]').fill(weko_admin_password)
    await page.get_by_role("button", name="ログイン").click()
    await expect(page.locator("button", has_text="Authorize application")).to_be_visible(timeout=transition_timeout)

    await page.locator("button", has_text="Authorize application").click()
    await logout_btn.is_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## サブメニューがプルダウンから「Log out」をクリックする

ログアウトしますか？の画面が表示される

In [ ]:
async def _step(page):
    if idp_name_1 == 'FakeCAS':
        logout_btn = page.locator('//a[@href = "/logout/"]')
        await logout_btn.click()
        await expect(logout_btn).not_to_be_visible(timeout=transition_timeout)
    else:
        logout_btn = page.get_by_role('button', name='ログアウト')
        await logout_btn.click()
        await expect(page.get_by_role("heading", name="ログアウトしますか？")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ログアウトしますか？の画面で「ログアウト」をクリックする

画面表示がログアウトしましたに切り替わる

In [ ]:
async def _step(page):
    if idp_name_1 == 'FakeCAS':
        return
    await page.get_by_role("button", name="ログアウト").click()
    await expect(page.get_by_role("heading", name="ログアウトしました")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのJAIRO Cloudのタブを閉じる

ブラウザのJAIRO Cloudのタブが閉じられる

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await page.locator('#projectNavFiles a').click()
    time.sleep(1)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## ブラウザでGRDMのタブに戻り、最上段右側のアカウント名をクリックする

サブメニューがプルダウンで表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//div[@class = "nav-profile-name"]').click()
    await expect(page.locator('a[href="/settings/"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## サブメニューがプルダウンから「Settings」をクリックする

Settingsの画面に遷移する

In [ ]:
async def _step(page):
    await page.locator('a[href="/settings/"]').click()
    await expect(page.locator('//*[text() = "アドオンアカウント構成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## Settingsの画面の左側メニューから「Configure add-on accounts」をクリックする

「Configure add-on accounts」のリストが表示される

In [ ]:
storage_id = "weko"
storage_name = "JAIRO Cloud"

async def _step(page):
    await page.locator('//*[text() = "アドオンアカウント構成"]').click()
    await expect(page.locator(f'//*[@src="/static/addons/{storage_id}/comicon.png"]/../*[@data-bind="text: properName" and text() = "{storage_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'//*[@src="/static/addons/{storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 「Configure add-on accounts」のリストの「JAIRO Cloud」の「Disconnect Account」をクリックする。

「アカウントを切断しますか？」のダイアログが表示される

In [ ]:
async def _step(page):
    await page.locator(f'//*[@src="/static/addons/{storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]').click()

    modal = page.locator('.modal-dialog')
    await expect(modal.get_by_role("heading", name="アカウントを切断しますか？")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アカウントを切断しますか？」のダイアログで「切断」ボタンをクリックする

「Configure add-on accounts」のリストのJAIRO Cloudが「Connect or Reauthorize Account」になる

In [ ]:
async def _step(page):
    modal = page.locator(".modal-dialog")
    await modal.get_by_role("button", name="切断").click()
    await expect(page.locator(f'//*[@src="/static/addons/{storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 最上段のメニューから「My Projects」をクリックする

マイプロジェクトの画面に遷移する

In [ ]:
async def _step(page):
    await page.goto(myproject_url)
    await expect(page.locator('//*[text() = "マイプロジェクト"]')).to_be_visible(timeout=transition_timeout)
    time.sleep(2)

await run_pw(_step)

## マイプロジェクトの画面で「TEST-メタデータ_未病データベース-ファイル操作-YYYYMMDD(本日の日付)」をクリックする

「TEST-メタデータ_未病データベース-ファイル操作-YYYYMMDD(本日の日付)」のプロジェクトダッシュボードに遷移する

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-nodetitle and text()="{rdm_project_name}"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, 'NII Storage')).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator(f'//div[@full_name = "{storage_name}"]//descendant::a[text() = "無効にする"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「JAIRO Cloud」の行を「無効にする」をクリックする。

「アドオンを無効にしますか？」のダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//div[@full_name = "{storage_name}"]//descendant::a[text() = "無効にする"]').click()
    modal = page.locator('.modal-dialog')
    await expect(modal.get_by_role("heading", name="アドオンを無効にしますか？")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを無効にしますか？」のダイアログで「無効にする」をクリックする

「アドオンを構成」のパネル内の「JAIRO Cloud」が消えること

In [ ]:
async def _step(page):
    modal = page.locator(".modal-dialog")
    await modal.get_by_role("button", name="無効にする").click()

    await expect(page.locator(f'#wekoScope h4.addon-title:has-text("{storage_name}")')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザの別タブで、GakuNinRDMの管理者画面を開く

GakuNinRDMの管理者画面のログイン画面が開く

In [ ]:
is_login = False

async def _step(page):
    await page.goto(admin_rdm_url)
    login_logo = page.locator('.login-logo')
    if await login_logo.is_visible():
        global is_login
        is_login = True
    else:
        print('User already logged in.')

    time.sleep(2)

await run_pw(_step)

## GakuNinRDMの管理者画面のログイン画面で、プルダウンから「GakuNin　RDM Idp」を選択して「選択」ボタンをクリック、管理者権限をもっているアカウント（email）とパスワードを入力して、Sign Inをクリックする

Username、Password入力画面が開く

In [ ]:
async def _step(page):
    if is_login:
        # IdPリストから所望のIdPを選択
        idplist = page.locator('//form[@id = "IdPList"]//input[@type = "text"]')
        await idplist.fill(idp_name_institutional_admin);
        await idplist.press('Enter');
        locator = page.locator('//*[contains(@class, "select") and normalize-space(text())="GakuNin RDM IdP"]').first
        await expect(locator).to_be_visible(timeout=transition_timeout)
        time.sleep(5)
        await locator.click()
    
        # 選択ボタンをクリック
        await page.locator('//input[@id = "wayf_submit_button"]').click()
    
        # アカウント入力欄が編集可能になったことを確認
        await expect(page.locator('#username')).to_be_editable()
    else:
        print('User already logged in.')

await run_pw(_step)

## Username、Passwordに、機関管理者権限のアカウントアカウントとパスワードを入力して、Loginをクリックする

RDM Adminのメインページが開く

In [ ]:
async def _step(page):
    if is_login:
        # メールアドレスとパスワードを入力
        await page.locator('#username').fill(idp_username_institutional_admin)
        await page.locator('#password').fill(idp_password_institutional_admin)
        await page.locator('//button[@type = "submit"]').click()
    
        # チェック「Ask me again at next login」が表示されることを確認 - 30秒以内に表示されることを期待
        await expect(page.locator('#_shib_idp_doNotRememberConsent')).to_be_enabled(timeout=transition_timeout)
    
        await page.locator('#_shib_idp_doNotRememberConsent').click()
        await expect(page.locator('#_shib_idp_doNotRememberConsent')).to_be_checked()
        await page.locator('//*[@name="_eventId_proceed"]').click()
    
        await expect(page.locator('a[href="/addons/"]')).to_be_enabled(timeout=transition_timeout)
    else:
        print('User already logged in.')

await run_pw(_step)

## 左側メニューよりRDM Addonsをクリックする

RDM Addonsのページが開くこと

In [ ]:
async def _step(page):
    await page.locator('a[href="/addons/"]').click()
    await page.wait_for_url("**/addons/**", timeout=transition_timeout)

    await expect(page.locator('h2:has-text("アドオン利用制御")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## RDM Addonsのページのリストから、JAIRO Cloudの「Delete Application」をクリックする

「JAIRO Cloudアプリケーションを削除しますか？」のダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator(
        f'em:has-text("{oauth_application_name}")'
    ).locator(
        'xpath=ancestor::div[contains(@class,"addon-auth-table")]/preceding-sibling::a[1]'
    ).click()

    modal = page.locator('.modal-dialog')
    await expect(modal.get_by_role("heading", name="JAIRO Cloudアプリケーションを削除しますか？")).to_be_visible(timeout=transition_timeout)
 
await run_pw(_step)


## 「JAIRO Cloudアプリケーションを削除しますか？」のダイアログの「切断」ボタンをクリックする

JAIRO Cloudに表示中の「Oauth Application ams-ir」が削除されること

In [ ]:
async def _step(page):
    modal = page.locator('.modal-dialog')
    await expect(modal.get_by_role("button", name="切断")).to_be_visible(timeout=transition_timeout)
    await modal.get_by_role("button", name="切断").click()

    addon_row = page.locator('tr.user-settings-addon-auth',has=page.locator('em', has_text=oauth_application_name))
    await expect(addon_row).to_have_count(0, timeout=transition_timeout)
 
await run_pw(_step)


## RDM Addonsのページのリストから、JAIRO Cloud選択しチェックボックスのチェックを外す

「JAIRO Cloudを禁止しますか？」のダイアログが表示されること

In [ ]:
is_check = False
async def _step(page):
    checkbox = page.locator(f'//input[@type = "checkbox" and @data-addon-full-name = "{storage_name}"]')
    if await checkbox.is_checked():
        await checkbox.click()
        modal = page.locator('.modal-dialog')
        await expect(modal.get_by_role("heading", name="JAIRO Cloudを禁止しますか？")).to_be_visible(timeout=transition_timeout)\

        global is_check
        is_check = True
    else:
        print('JAIRO Cloud already disabled for this institution')

await run_pw(_step)

## 「JAIRO Cloudを禁止しますか？」のダイアログの入力欄にダイアログで指示された、ワンタイムパスワードを入力して「禁止」ボタンをクリックする

RDM Addonsのページのリストの、JAIRO Cloudチェックが外れる事

In [ ]:
async def _step(page):
    if is_check:
        modal = page.locator('.modal-dialog')
        confirm_key = (await modal.locator('//h4[text() = "JAIRO Cloudを禁止しますか？"]/../..//strong').inner_text()).strip()
        await modal.locator('//input[@id = "wekoDeleteKey"]').fill(confirm_key)
        await modal.locator('//button[text() = "禁止"]').click()

        checkbox = page.locator(f'//input[@type = "checkbox" and @data-addon-full-name = "{storage_name}"]')
        await expect(checkbox).not_to_be_checked(timeout=transition_timeout)
        time.sleep(2)
    else:
        print('JAIRO Cloud already disabled for this institution')

await run_pw(_step)

## ブラウザのGakuNinRDMの管理者画面のタブを閉じる

ブラウザのGakuNinRDMの管理者画面のタブが閉じられる

In [ ]:
async def _step(page):
    new_tab = await page.context.new_page()
    await page.close()
    assert page.is_closed()
    assert not new_tab.is_closed()

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}